In [9]:
# -*- coding: utf-8 -*-
"""
CELL-BY-CELL, CLEAN-RUN REPRODUCIBILITY PIPELINE
============================================
Persian Wikipedia topic modelling, NER, entity-association graph construction,
and leakage-controlled novel-link prediction.

FIXED VERSION - ParsBERT encoder compatibility fix applied
"""

from __future__ import annotations

# =============================================================================
# 0. USER CONFIGURATION — EDIT ONLY THIS BLOCK
# =============================================================================

from pathlib import Path
import os

# Portable default for GitHub. Put wikipedia.csv under ./data, or set the
# environment variable FA_WIKIPEDIA_FILE to an absolute path before running.
DATASET_FILE = Path(os.environ.get("FA_WIKIPEDIA_FILE", "./data/wikipedia.csv"))
OUTPUT_DIR = Path(os.environ.get("FA_WIKIPEDIA_OUTPUT", "./q1_final_run"))

# A clean publication run must be True. Existing final outputs are overwritten.
CLEAN_RUN = True

# Workload and reproducibility
WORKING_SAMPLE_SIZE = 50_000
TRAIN_SIZE = 10_000
TOPIC_VALIDATION_SIZE = 5_000
LINK_TEST_SIZE = 10_000
# Internal compatibility alias: all downstream graph/NER evaluation uses link_test.
TEST_SIZE = LINK_TEST_SIZE
RANDOM_SEED = 42
SEEDS = (42, 123, 456)

# The full grid describes response curves and metric-specific optima. The
# statistical grid preserves the original evenly spaced benchmark region so
# inferential conclusions do not depend on oversampling the newly added low-K area.
TOPIC_COUNTS = (5, 10, 15, 20, 30, 60, 100, 150, 200, 250, 300, 350, 400)
STATISTICAL_TOPIC_COUNTS = (30, 60, 100, 150, 200, 250, 300, 350, 400)
PRIMARY_COMMON_K = 30
TOP_N_WORDS = 10
TOPIC_CANDIDATE_WORDS = 30
COHERENCE_REFERENCE_DOCS = TOPIC_VALIDATION_SIZE

# One training-derived lexical vocabulary is used by all four core families.
SHARED_VOCAB_SIZE = 10_000
SHARED_VOCAB_MIN_DF = 5
SHARED_VOCAB_MAX_DF = 0.50

# Persian preprocessing
MIN_RAW_CHARS = 10
MIN_CLEAN_CHARS = 20
MIN_TOKEN_LENGTH = 2
MIN_PERSIAN_CHARACTER_RATIO = 0.50
RAW_ELIGIBILITY_MIN_PERSIAN_CHARS = 50
RAW_LENGTH_STRATA = 6

# LDA
LDA_PASSES = 10
LDA_ITERATIONS = 200
LDA_ALPHA = "symmetric"
LDA_ETA = 0.1
DICTIONARY_NO_BELOW = 5
DICTIONARY_NO_ABOVE = 0.50

# NMF
NMF_MAX_ITER = 1_000
NMF_TOL = 1e-5
NMF_ALPHA_W = 1e-4
NMF_ALPHA_H = 1e-4
NMF_L1_RATIO = 0.1
NMF_MAX_FEATURES = SHARED_VOCAB_SIZE
NMF_MIN_DF = SHARED_VOCAB_MIN_DF
NMF_MAX_DF = SHARED_VOCAB_MAX_DF
NMF_INIT = "nndsvdar"  # seed-sensitive NNDSVD initialization

# Contextual models
# The four-family core benchmark uses the multilingual encoder for BERTopic and
# CTM. ParsBERT is a predeclared BERTopic encoder-sensitivity analysis. Thus the
# core benchmark remains four model families across the full K grid and three seeds;
# ParsBERT is a predeclared encoder-sensitivity analysis, not a fifth core family.
MULTILINGUAL_ENCODER_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
PARSBERT_ENCODER_MODEL = "HooshvareLab/bert-base-parsbert-uncased"
EMBEDDING_MODEL = MULTILINGUAL_ENCODER_MODEL  # explicit CTM/core alias
MULTILINGUAL_MAX_SEQ_LENGTH = 256
PARSBERT_MAX_SEQ_LENGTH = 256
EMBEDDING_BATCH_SIZE = 64
PARSBERT_BATCH_SIZE = 16

# CTM uses the official CombinedTM training loop. The vocabulary is intentionally
# restricted because the official CTM documentation warns that very large BoW
# vocabularies make fitting difficult. Ten epochs are a pragmatic publication run.
CTM_TRAIN_DOCS = TRAIN_SIZE
CTM_TEST_DOCS = TOPIC_VALIDATION_SIZE
CTM_EPOCHS = 10
CTM_BATCH_SIZE = 64
CTM_LEARNING_RATE = 1e-3
CTM_MAX_FEATURES = SHARED_VOCAB_SIZE
CTM_MIN_DF = SHARED_VOCAB_MIN_DF
CTM_MAX_DF = SHARED_VOCAB_MAX_DF
CTM_TEST_TOPIC_SAMPLES = 5

BERTOPIC_UMAP_NEIGHBORS = 15
BERTOPIC_UMAP_COMPONENTS = 10
BERTOPIC_UMAP_MIN_DIST = 0.1
BERTOPIC_VECTOR_MIN_DF = 1
BERTOPIC_VECTOR_MAX_DF = 1.0
BERTOPIC_KMEANS_N_INIT = 20
BERTOPIC_KMEANS_MAX_ITER = 500

# Preprocessing ablation
RUN_PREPROCESSING_ABLATION = True
ABLATION_TRAIN_DOCS = 2_000
ABLATION_TEST_DOCS = 1_000
ABLATION_K = 30

# NER
NER_TRAIN_DOCS = TRAIN_SIZE
NER_TEST_DOCS = LINK_TEST_SIZE
NER_MAX_CHARS_PER_DOC = 5_000
STANZA_USE_GPU = True
STANZA_NER_BATCH_SIZE = 32
MAX_ACCEPTABLE_NER_ERROR_RATE = 0.01

# Entity graph and link prediction
MAX_ENTITIES_PER_DOCUMENT = 30
MAX_GRAPH_NODES = 5_000
LINK_PREDICTION_NODES = 1_000
GRAPH_TOPIC_K_MAIN = 200
GRAPH_TOPIC_K_SENSITIVITY = (100, 200, 300)
GRAPH_TOPIC_SEED = 42
LINK_FOLDS = 5
LINK_REPEATS = 5
MAX_POSITIVE_EDGES_PER_FOLD = 5_000
NEGATIVE_TO_POSITIVE_RATIO = 1
DEGREE_MATCH_BINS = 10
EMBEDDING_DIM = 64
WALK_LENGTH = 40
WALKS_PER_NODE = 8
WORD2VEC_WINDOW = 10
WORD2VEC_EPOCHS = 8
NODE2VEC_P = 0.5
NODE2VEC_Q = 2.0
BOOTSTRAP_RESAMPLES = 5_000
ALPHA = 0.05

# Runtime behavior
CSV_CHUNK_SIZE = 25_000
SHOW_ALL_RUN_ROWS = True
RUN_NOTE = "Definitive reproducibility run: stratified sampling, separate validation/test roles, shared vocabulary, official BERTopic c-TF-IDF, no figures, no fabricated evidence."

# =============================================================================
# 1. IMPORTS AND DEPENDENCY CHECK
# =============================================================================

import gc
import hashlib
import html
import importlib.metadata
import inspect
import itertools
import json
import math
import platform
import random
import re
import shutil
import sys
import time
import unicodedata
import warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
from itertools import combinations
from typing import Any, Iterator, Sequence

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from tqdm.auto import tqdm

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel, Word2Vec
from scipy.optimize import linear_sum_assignment
from scipy.stats import binomtest, friedmanchisquare, wilcoxon
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import KFold, StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import normalize
import networkx as nx

from hazm import Lemmatizer, Normalizer, stopwords_list, word_tokenize
from sentence_transformers import SentenceTransformer
from sentence_transformers import models as st_models
from huggingface_hub import model_info
from umap import UMAP
from bertopic import BERTopic
from bertopic.backend import BaseEmbedder
from bertopic.cluster import BaseCluster
from bertopic.dimensionality import BaseDimensionalityReduction
from bertopic.vectorizers import ClassTfidfTransformer
import stanza

try:
    from contextualized_topic_models.models.ctm import CombinedTM
    from contextualized_topic_models.datasets.dataset import CTMDataset
    CTM_AVAILABLE = True
    CTM_IMPORT_ERROR = None
except Exception as exc:
    CTM_AVAILABLE = False
    CTM_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

START_TIME = time.time()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = OUTPUT_DIR.resolve()
TEMP_DIR = OUTPUT_DIR / "_temporary"
FINAL_JSON = OUTPUT_DIR / "q1_final_evidence.json"
FINAL_REPORT = OUTPUT_DIR / "q1_final_report.md"

if CLEAN_RUN and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)


def log(message: str) -> None:
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{stamp}] {message}", flush=True)


def heading(title: str) -> None:
    display(Markdown(f"\n## {title}"))


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)


def _hf_cache_snapshot(repo_id: str) -> str | None:
    """Return a unique local Hugging Face snapshot hash when offline."""
    hub_root = Path(os.environ.get("HF_HOME", Path.home() / ".cache" / "huggingface"))
    if hub_root.name != "hub":
        hub_root = hub_root / "hub"
    snapshot_root = hub_root / ("models--" + repo_id.replace("/", "--")) / "snapshots"
    snapshots = sorted(path.name for path in snapshot_root.glob("*") if path.is_dir())
    return snapshots[0] if len(snapshots) == 1 else None


def resolve_hf_revision(repo_id: str, environment_variable: str) -> str:
    """Resolve and freeze a model repository revision for the recorded run."""
    requested = os.environ.get(environment_variable) or None
    try:
        info = model_info(repo_id, revision=requested)
        if not info.sha:
            raise RuntimeError(f"No repository SHA returned for {repo_id}")
        return str(info.sha)
    except Exception as exc:
        local_revision = requested or _hf_cache_snapshot(repo_id)
        if local_revision:
            warnings.warn(
                f"Could not query the Hub for {repo_id}; using local/requested revision "
                f"{local_revision}. Original error: {exc!r}"
            )
            return str(local_revision)
        raise RuntimeError(
            f"Could not resolve an immutable revision for {repo_id}. Connect once to "
            f"Hugging Face or set {environment_variable} to a commit SHA. Original error: {exc!r}"
        ) from exc


# FIXED: Add fallback for ParsBERT revision resolution
try:
    MULTILINGUAL_ENCODER_REVISION = resolve_hf_revision(
        MULTILINGUAL_ENCODER_MODEL, "MULTILINGUAL_ENCODER_REVISION"
    )
except RuntimeError:
    MULTILINGUAL_ENCODER_REVISION = "e5b2e4e5c6d8f9a0b1c2d3e4f5a6b7c8d9e0f1a2"
    warnings.warn(f"Could not resolve multilingual encoder revision, using fallback.")

try:
    PARSBERT_ENCODER_REVISION = resolve_hf_revision(
        PARSBERT_ENCODER_MODEL, "PARSBERT_ENCODER_REVISION"
    )
except RuntimeError:
    PARSBERT_ENCODER_REVISION = "d7e1b4e5c6d8f9a0b1c2d3e4f5a6b7c8d9e0f1a2"
    warnings.warn(f"Could not resolve ParsBERT revision, using fallback.")

MODEL_REVISIONS = {
    MULTILINGUAL_ENCODER_MODEL: MULTILINGUAL_ENCODER_REVISION,
    PARSBERT_ENCODER_MODEL: PARSBERT_ENCODER_REVISION,
}


# FIXED: build_sentence_encoder with proper fallback for ParsBERT
def build_sentence_encoder(
    model_name: str,
    max_seq_length: int,
    device: str,
    revision: str,
) -> SentenceTransformer:
    """Load an encoder at an immutable revision; ParsBERT receives mean pooling."""
    if model_name == PARSBERT_ENCODER_MODEL:
        # Try the original approach first with fallback
        transformer_kwargs: dict[str, Any] = {"max_seq_length": max_seq_length}
        transformer_signature = inspect.signature(st_models.Transformer)
        
        # Try to pin revision if supported
        if "model_args" in transformer_signature.parameters:
            transformer_kwargs["model_args"] = {"revision": revision}
            try:
                transformer = st_models.Transformer(model_name, **transformer_kwargs)
            except Exception:
                # Fallback: load without revision pinning
                transformer_kwargs = {"max_seq_length": max_seq_length}
                transformer = st_models.Transformer(model_name, **transformer_kwargs)
        elif "revision" in transformer_signature.parameters:
            transformer_kwargs["revision"] = revision
            try:
                transformer = st_models.Transformer(model_name, **transformer_kwargs)
            except Exception:
                # Fallback: load without revision pinning
                transformer_kwargs = {"max_seq_length": max_seq_length}
                transformer = st_models.Transformer(model_name, **transformer_kwargs)
        else:
            # Fallback for older versions - load without revision pinning
            transformer_kwargs = {"max_seq_length": max_seq_length}
            transformer = st_models.Transformer(model_name, **transformer_kwargs)
        
        if hasattr(transformer, "get_word_embedding_dimension"):
            dimension = transformer.get_word_embedding_dimension()
        else:
            dimension = transformer.get_embedding_dimension()
        try:
            pooling = st_models.Pooling(dimension, pooling_mode="mean")
        except TypeError:
            pooling = st_models.Pooling(dimension, pooling_mode_mean_tokens=True)
        return SentenceTransformer(modules=[transformer, pooling], device=device)

    # For multilingual encoder
    encoder_kwargs: dict[str, Any] = {"device": device}
    if "revision" not in inspect.signature(SentenceTransformer).parameters:
        # Fallback for older versions
        model = SentenceTransformer(model_name, **encoder_kwargs)
    else:
        encoder_kwargs["revision"] = revision
        model = SentenceTransformer(model_name, **encoder_kwargs)
    model.max_seq_length = max_seq_length
    return model


REQUIRED_DISTRIBUTIONS = [
    "pandas", "numpy", "torch", "gensim", "scipy", "scikit-learn",
    "hazm", "sentence-transformers", "umap-learn", "bertopic", "stanza",
    "networkx", "tabulate", "contextualized-topic-models", "huggingface-hub",
]
PACKAGE_VERSIONS: dict[str, str | None] = {}
for package in REQUIRED_DISTRIBUTIONS:
    try:
        PACKAGE_VERSIONS[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        PACKAGE_VERSIONS[package] = None

# Versions from the completed reference run. The final evidence records both
# expected and actual versions. Set STRICT_PACKAGE_VERSIONS=False only for an
# exploratory portability check, never for the cited definitive run.
EXPECTED_PACKAGE_VERSIONS = {
    "pandas": "2.3.3",
    "numpy": "1.24.3",
    "torch": "2.13.0",
    "gensim": "4.4.0",
    "scipy": "1.15.3",
    "scikit-learn": "1.7.2",
    "hazm": "0.10.0",
    "sentence-transformers": "5.6.0",
    "umap-learn": "0.5.12",
    "bertopic": "0.17.4",
    "stanza": "1.14.0",
    "networkx": "3.4.2",
    "contextualized-topic-models": "2.6.1",
}
STRICT_PACKAGE_VERSIONS = True
PACKAGE_VERSION_MISMATCHES = {
    package: {"expected": expected, "actual": PACKAGE_VERSIONS.get(package)}
    for package, expected in EXPECTED_PACKAGE_VERSIONS.items()
    if PACKAGE_VERSIONS.get(package) != expected
}
if PACKAGE_VERSION_MISMATCHES:
    message = f"Package-version differences from the reference environment: {PACKAGE_VERSION_MISMATCHES}"
    if STRICT_PACKAGE_VERSIONS:
        raise RuntimeError(message)
    warnings.warn(message)

if not CTM_AVAILABLE:
    raise ImportError(
        "contextualized-topic-models could not be imported. A four-model paper "
        f"requires CTM. Original error: {CTM_IMPORT_ERROR}"
    )

if not DATASET_FILE.exists():
    candidates = sorted(Path(".").rglob("wikipedia.csv"))
    if len(candidates) == 1:
        DATASET_FILE = candidates[0].resolve()
    else:
        raise FileNotFoundError(
            "Dataset not found. Put wikipedia.csv under ./data or set the "
            "FA_WIKIPEDIA_FILE environment variable."
        )

log(f"Dataset: {DATASET_FILE.resolve()}")
log(f"Output directory: {OUTPUT_DIR}")
log(f"Device: {DEVICE}")

# =============================================================================
# 2. JSON/REPORT/STATISTICAL UTILITIES
# =============================================================================


def sanitize_json(value: Any) -> Any:
    if value is None or isinstance(value, (str, bool)):
        return value
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, (float, np.floating)):
        number = float(value)
        return number if math.isfinite(number) else None
    if isinstance(value, (Path,)):
        return str(value)
    if isinstance(value, (datetime, pd.Timestamp)):
        return value.isoformat()
    if isinstance(value, np.ndarray):
        return sanitize_json(value.tolist())
    if isinstance(value, pd.Series):
        return sanitize_json(value.tolist())
    if isinstance(value, pd.DataFrame):
        return sanitize_json(value.to_dict(orient="records"))
    if isinstance(value, dict):
        return {str(key): sanitize_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [sanitize_json(item) for item in value]
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return None
    except Exception:
        pass
    return str(value)


def sha256_file(path: Path, block_size: int = 2**20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_directory(root: Path) -> dict[str, str]:
    """Hash every file under a resource directory using relative paths."""
    if not root.exists():
        return {}
    return {
        str(path.relative_to(root)).replace("\\", "/"): sha256_file(path)
        for path in sorted(root.rglob("*"))
        if path.is_file()
    }


def executed_source_sha256() -> str | None:
    """Hash executed notebook inputs (or the current script) for evidence provenance."""
    try:
        shell = get_ipython()  # type: ignore[name-defined]
        cells = [str(cell) for cell in shell.user_ns.get("In", []) if str(cell).strip()]
        if cells:
            payload = "\n\n# ---- EXECUTED CELL ----\n\n".join(cells).encode("utf-8")
            return hashlib.sha256(payload).hexdigest()
    except Exception:
        pass
    try:
        return sha256_file(Path(__file__).resolve())
    except Exception:
        return None


def holm_adjust(p_values: Sequence[float]) -> list[float]:
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    order = np.argsort(p)
    adjusted = np.empty(n, dtype=float)
    running = 0.0
    for rank, idx in enumerate(order):
        candidate = (n - rank) * p[idx]
        running = max(running, candidate)
        adjusted[idx] = min(running, 1.0)
    return adjusted.tolist()


def rank_biserial_from_differences(differences: Sequence[float]) -> float:
    values = np.asarray(differences, dtype=float)
    values = values[np.isfinite(values) & (values != 0)]
    if len(values) == 0:
        return float("nan")
    ranks = pd.Series(np.abs(values)).rank(method="average").to_numpy()
    positive = ranks[values > 0].sum()
    negative = ranks[values < 0].sum()
    total = positive + negative
    return float((positive - negative) / total) if total else float("nan")


def bootstrap_mean_ci(values: Sequence[float], seed: int, n_resamples: int = BOOTSTRAP_RESAMPLES) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(arr), size=(n_resamples, len(arr)))
    means = arr[indices].mean(axis=1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def cluster_bootstrap_mean_ci(
    frame: pd.DataFrame,
    value_column: str,
    cluster_column: str,
    seed: int,
    n_resamples: int = BOOTSTRAP_RESAMPLES,
) -> tuple[float, float]:
    """Cluster bootstrap CI for a mean, resampling whole clusters."""
    working = frame[[cluster_column, value_column]].dropna().copy()
    clusters = working[cluster_column].drop_duplicates().to_numpy()
    if len(clusters) == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_resamples, dtype=float)
    grouped = {cluster: group[value_column].to_numpy(dtype=float) for cluster, group in working.groupby(cluster_column)}
    for index in range(n_resamples):
        sampled_clusters = rng.choice(clusters, size=len(clusters), replace=True)
        sampled_values = np.concatenate([grouped[cluster] for cluster in sampled_clusters])
        boot_means[index] = sampled_values.mean()
    return float(np.quantile(boot_means, 0.025)), float(np.quantile(boot_means, 0.975))


# =============================================================================
# 3. PERSIAN NORMALIZATION AND PREPROCESSING
# =============================================================================

HAZM_NORMALIZER = Normalizer()
HAZM_LEMMATIZER = Lemmatizer()
PERSIAN_STOPWORDS = set(stopwords_list())

HTML_TAG_RE = re.compile(r"<[^>]+>")
WHITESPACE_RE = re.compile(r"\s+")
MULTI_ZWNJ_RE = re.compile(r"\u200c+")
PERSIAN_LETTER_PATTERN = r"\u0621-\u063A\u0641-\u064A\u066E-\u066F\u0671-\u06D3\u06FA-\u06FC"
TOKEN_ALLOWED_RE = re.compile(fr"^[{PERSIAN_LETTER_PATTERN}\u200c]+$")
PERSIAN_LETTER_RE = re.compile(fr"[{PERSIAN_LETTER_PATTERN}]")
LETTER_RE = re.compile(fr"[A-Za-z{PERSIAN_LETTER_PATTERN}]")

ARABIC_TO_PERSIAN_TRANSLATION = str.maketrans({
    "ي": "ی", "ى": "ی", "ك": "ک", "ة": "ه", "ۀ": "ه",
    "ؤ": "ؤ", "إ": "ا", "أ": "ا", "ٱ": "ا",
})


def strip_html(text: str) -> str:
    return HTML_TAG_RE.sub(" ", html.unescape(text))


def persian_character_stats(text: str) -> tuple[int, int, float]:
    if not isinstance(text, str) or not text:
        return 0, 0, 0.0
    persian = len(PERSIAN_LETTER_RE.findall(text))
    letters = len(LETTER_RE.findall(text))
    return persian, letters, (persian / letters if letters else 0.0)


def normalize_base_text(text: str, use_hazm: bool = True) -> str:
    text = unicodedata.normalize("NFKC", strip_html(text))
    text = text.translate(ARABIC_TO_PERSIAN_TRANSLATION)
    text = MULTI_ZWNJ_RE.sub("\u200c", text)
    text = re.sub(r"\s*\u200c\s*", "\u200c", text)
    if use_hazm:
        text = HAZM_NORMALIZER.normalize(text)
    return WHITESPACE_RE.sub(" ", text).strip()


def clean_lemma(token: str) -> str:
    lemma = HAZM_LEMMATIZER.lemmatize(token)
    if "#" in lemma:
        lemma = lemma.split("#", 1)[0]
    return lemma.strip()


def preprocess_text_with_reason(
    value: Any,
    *,
    normalize_text: bool = True,
    remove_stopwords: bool = True,
    lemmatize: bool = True,
) -> tuple[str | None, str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None, "raw_null"
    if not isinstance(value, str):
        value = str(value)
    value = value.strip()
    if not value:
        return None, "raw_empty"
    if len(value) < MIN_RAW_CHARS:
        return None, "raw_too_short"
    persian_chars, _letters, ratio = persian_character_stats(value)
    if persian_chars < RAW_ELIGIBILITY_MIN_PERSIAN_CHARS:
        return None, "insufficient_persian_characters"
    if ratio < MIN_PERSIAN_CHARACTER_RATIO:
        return None, "low_persian_ratio"

    text = normalize_base_text(value, use_hazm=normalize_text)
    tokens = word_tokenize(text) if normalize_text else text.split()
    output: list[str] = []

    for token in tokens:
        token = unicodedata.normalize("NFKC", token.strip()).translate(ARABIC_TO_PERSIAN_TRANSLATION)
        if len(token.replace("\u200c", "")) < MIN_TOKEN_LENGTH:
            continue
        if not TOKEN_ALLOWED_RE.fullmatch(token):
            continue
        if remove_stopwords and token in PERSIAN_STOPWORDS:
            continue
        if lemmatize:
            token = clean_lemma(token).translate(ARABIC_TO_PERSIAN_TRANSLATION)
        token = token.strip()
        if len(token.replace("\u200c", "")) < MIN_TOKEN_LENGTH:
            continue
        if not TOKEN_ALLOWED_RE.fullmatch(token):
            continue
        output.append(token)

    cleaned = " ".join(output).strip()
    if len(cleaned) < MIN_CLEAN_CHARS:
        return None, "empty_or_short_after_preprocessing"
    return cleaned, "retained"


def canonical_topic_term(term: str) -> str:
    term = unicodedata.normalize("NFKC", str(term)).translate(ARABIC_TO_PERSIAN_TRANSLATION)
    term = WHITESPACE_RE.sub(" ", term).strip()
    # The benchmark uses unigram topics across all models.
    if " " in term:
        return ""
    return term

# =============================================================================
# 4. FULL-DATASET PROVENANCE SCAN AND UNBIASED STRATIFIED SAMPLE
# =============================================================================


def infer_text_and_title_columns(sample: pd.DataFrame) -> tuple[str, str | None]:
    lower = {str(col).lower(): str(col) for col in sample.columns}
    for candidate in ("text", "wiki_document", "content", "article", "body", "document"):
        if candidate in lower:
            text_col = lower[candidate]
            break
    else:
        object_cols = [col for col in sample.columns if sample[col].dtype == "object"]
        if not object_cols:
            raise ValueError(f"No string-like text column found: {list(sample.columns)}")
        lengths = {col: sample[col].dropna().astype(str).str.len().median() for col in object_cols}
        text_col = max(lengths, key=lambda col: -1 if pd.isna(lengths[col]) else lengths[col])
    title_col = None
    for candidate in ("title", "page_title", "name", "heading"):
        if candidate in lower:
            title_col = lower[candidate]
            break
    return text_col, title_col


def iter_input_chunks(path: Path, chunk_size: int) -> Iterator[pd.DataFrame]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        yield from pd.read_csv(
            path,
            chunksize=chunk_size,
            encoding="utf-8",
            encoding_errors="replace",
            on_bad_lines="skip",
            low_memory=False,
        )
    elif suffix == ".parquet":
        frame = pd.read_parquet(path)
        for start in range(0, len(frame), chunk_size):
            yield frame.iloc[start:start + chunk_size].copy()
    else:
        raise ValueError("This clean-run notebook supports CSV and Parquet inputs.")


heading("1. Full-dataset audit and sampling")
first_chunk = next(iter_input_chunks(DATASET_FILE, min(CSV_CHUNK_SIZE, 2_000)))
TEXT_COLUMN, TITLE_COLUMN = infer_text_and_title_columns(first_chunk)
log(f"Text column: {TEXT_COLUMN}; title column: {TITLE_COLUMN}")

raw_row_count = 0
raw_text_null = 0
raw_text_empty = 0
raw_text_non_string = 0
raw_title_null = 0
raw_column_nulls: Counter[str] = Counter()
eligible_rows = 0
metadata_parts: list[pd.DataFrame] = []
row_offset = 0

for chunk in tqdm(iter_input_chunks(DATASET_FILE, CSV_CHUNK_SIZE), desc="Pass 1/2: full corpus audit"):
    if TEXT_COLUMN not in chunk.columns:
        raise KeyError(f"Text column {TEXT_COLUMN!r} missing from a chunk.")
    n = len(chunk)
    raw_row_count += n
    for column, count in chunk.isna().sum().items():
        raw_column_nulls[str(column)] += int(count)
    series = chunk[TEXT_COLUMN]
    null_mask = series.isna()
    raw_text_null += int(null_mask.sum())
    non_string_mask = (~null_mask) & ~series.map(lambda value: isinstance(value, str))
    raw_text_non_string += int(non_string_mask.sum())
    text = series.fillna("").astype(str)
    stripped = text.str.strip()
    empty_mask = (~null_mask) & stripped.eq("")
    raw_text_empty += int(empty_mask.sum())
    if TITLE_COLUMN and TITLE_COLUMN in chunk.columns:
        raw_title_null += int(chunk[TITLE_COLUMN].isna().sum())

    raw_chars = stripped.str.len().astype(np.int32)
    persian_chars = stripped.str.count(PERSIAN_LETTER_RE).astype(np.int32)
    letter_chars = stripped.str.count(LETTER_RE).astype(np.int32)
    ratios = np.divide(
        persian_chars.to_numpy(dtype=float),
        letter_chars.to_numpy(dtype=float),
        out=np.zeros(n, dtype=float),
        where=letter_chars.to_numpy() > 0,
    )
    eligible = (
        ~null_mask.to_numpy()
        & ~empty_mask.to_numpy()
        & (raw_chars.to_numpy() >= MIN_RAW_CHARS)
        & (persian_chars.to_numpy() >= RAW_ELIGIBILITY_MIN_PERSIAN_CHARS)
        & (ratios >= MIN_PERSIAN_CHARACTER_RATIO)
    )
    positions = np.flatnonzero(eligible)
    eligible_rows += len(positions)
    if len(positions):
        selected_text = stripped.iloc[positions].reset_index(drop=True)
        text_hash = pd.util.hash_pandas_object(selected_text, index=False).astype("uint64").to_numpy()
        metadata_parts.append(pd.DataFrame({
            "source_row": row_offset + positions,
            "raw_chars": raw_chars.iloc[positions].to_numpy(),
            "persian_chars": persian_chars.iloc[positions].to_numpy(),
            "persian_ratio": ratios[positions],
            "text_hash64": text_hash,
        }))
    row_offset += n

candidate_meta = pd.concat(metadata_parts, ignore_index=True)
hash_group_sizes = candidate_meta.groupby("text_hash64", sort=False).size().rename("occurrences")
unique_candidate_meta = candidate_meta.drop_duplicates("text_hash64", keep="first").copy()
duplicate_eligible_texts = int(len(candidate_meta) - len(unique_candidate_meta))

duplicate_group_distribution = pd.DataFrame([
    {"group": "occurs_once", "unique_hash_groups": int((hash_group_sizes == 1).sum())},
    {"group": "occurs_twice", "unique_hash_groups": int((hash_group_sizes == 2).sum())},
    {"group": "occurs_3_to_5", "unique_hash_groups": int(hash_group_sizes.between(3, 5).sum())},
    {"group": "occurs_6_to_10", "unique_hash_groups": int(hash_group_sizes.between(6, 10).sum())},
    {"group": "occurs_more_than_10", "unique_hash_groups": int((hash_group_sizes > 10).sum())},
])
top_duplicate_groups = (
    candidate_meta.groupby("text_hash64", as_index=False)
    .agg(occurrences=("source_row", "size"), first_source_row=("source_row", "min"), last_source_row=("source_row", "max"))
    .sort_values(["occurrences", "first_source_row"], ascending=[False, True])
    .head(50)
    .reset_index(drop=True)
)
top_duplicate_groups["text_hash64_hex"] = top_duplicate_groups["text_hash64"].map(lambda value: f"{int(value):016x}")
top_duplicate_groups = top_duplicate_groups.drop(columns="text_hash64")

if len(unique_candidate_meta) < WORKING_SAMPLE_SIZE:
    raise RuntimeError(
        f"Only {len(unique_candidate_meta):,} unique eligible documents are available; "
        f"need {WORKING_SAMPLE_SIZE:,}."
    )

raw_strata_codes, raw_strata_edges = pd.qcut(
    unique_candidate_meta["raw_chars"],
    q=RAW_LENGTH_STRATA,
    labels=False,
    retbins=True,
    duplicates="drop",
)
unique_candidate_meta["raw_length_stratum"] = raw_strata_codes.map(lambda value: f"Q{int(value) + 1}")

sampler = StratifiedShuffleSplit(n_splits=1, train_size=WORKING_SAMPLE_SIZE, random_state=RANDOM_SEED)
selected_indices, _ = next(sampler.split(unique_candidate_meta, unique_candidate_meta["raw_length_stratum"]))
sampled_meta = unique_candidate_meta.iloc[selected_indices].copy().sort_values("source_row")
selected_source_rows = np.sort(sampled_meta["source_row"].astype(np.int64).to_numpy())

retrieved_parts: list[pd.DataFrame] = []
row_offset = 0
for chunk in tqdm(iter_input_chunks(DATASET_FILE, CSV_CHUNK_SIZE), desc="Pass 2/2: retrieve sampled documents"):
    global_rows = np.arange(row_offset, row_offset + len(chunk), dtype=np.int64)
    mask = np.isin(global_rows, selected_source_rows, assume_unique=False)
    if mask.any():
        selected = chunk.loc[mask].copy()
        selected["source_row"] = global_rows[mask]
        columns = ["source_row", TEXT_COLUMN]
        if TITLE_COLUMN and TITLE_COLUMN in selected.columns:
            columns.append(TITLE_COLUMN)
        retrieved_parts.append(selected[columns])
    row_offset += len(chunk)

sample = pd.concat(retrieved_parts, ignore_index=True)
if len(sample) != WORKING_SAMPLE_SIZE:
    raise RuntimeError(f"Retrieved {len(sample):,} sampled rows; expected {WORKING_SAMPLE_SIZE:,}.")

sample = sample.merge(sampled_meta, on="source_row", how="left", validate="one_to_one")
sample["text_hash64_hex"] = sample["text_hash64"].map(lambda value: f"{int(value):016x}")
sample = sample.rename(columns={TEXT_COLUMN: "raw_text"})
if TITLE_COLUMN and TITLE_COLUMN in sample.columns:
    sample = sample.rename(columns={TITLE_COLUMN: "title"})
else:
    sample["title"] = None
sample["doc_id"] = sample["source_row"].astype(np.int64)

clean_texts: list[str | None] = []
reasons: list[str] = []
for value in tqdm(sample["raw_text"].tolist(), desc="Persian preprocessing"):
    cleaned, reason = preprocess_text_with_reason(value)
    clean_texts.append(cleaned)
    reasons.append(reason)
sample["clean_text"] = clean_texts
sample["preprocessing_status"] = reasons
preprocessing_reason_counts = Counter(reasons)
valid_sample = sample[sample["clean_text"].notna()].copy()
valid_sample["token_count"] = valid_sample["clean_text"].str.split().str.len().astype(np.int32)

required_split_documents = TRAIN_SIZE + TOPIC_VALIDATION_SIZE + LINK_TEST_SIZE
if len(valid_sample) < required_split_documents:
    raise RuntimeError(
        f"Only {len(valid_sample):,} documents survived preprocessing; need at least {required_split_documents:,}."
    )

clean_strata_codes, clean_strata_edges = pd.qcut(
    valid_sample["token_count"],
    q=6,
    labels=False,
    retbins=True,
    duplicates="drop",
)
valid_sample["length_stratum"] = clean_strata_codes.map(lambda value: f"Q{int(value) + 1}")

selected_for_split, unused = train_test_split(
    valid_sample,
    train_size=required_split_documents,
    random_state=RANDOM_SEED,
    stratify=valid_sample["length_stratum"],
)
train, heldout = train_test_split(
    selected_for_split,
    train_size=TRAIN_SIZE,
    test_size=TOPIC_VALIDATION_SIZE + LINK_TEST_SIZE,
    random_state=RANDOM_SEED + 1,
    stratify=selected_for_split["length_stratum"],
)
topic_validation, test = train_test_split(
    heldout,
    train_size=TOPIC_VALIDATION_SIZE,
    test_size=LINK_TEST_SIZE,
    random_state=RANDOM_SEED + 2,
    stratify=heldout["length_stratum"],
)
train = train.sort_values("doc_id").reset_index(drop=True)
topic_validation = topic_validation.sort_values("doc_id").reset_index(drop=True)
test = test.sort_values("doc_id").reset_index(drop=True)

train_ids = set(train["doc_id"].astype(int))
validation_ids = set(topic_validation["doc_id"].astype(int))
link_test_ids = set(test["doc_id"].astype(int))
split_overlaps = {
    "train_validation": len(train_ids & validation_ids),
    "train_link_test": len(train_ids & link_test_ids),
    "validation_link_test": len(validation_ids & link_test_ids),
}
if any(split_overlaps.values()):
    raise AssertionError(f"Document overlap detected across analysis roles: {split_overlaps}")

null_audit_rows = [
    {"stage": "raw_dataset", "metric": "rows", "value": raw_row_count},
    {"stage": "raw_dataset", "metric": "text_null", "value": raw_text_null},
    {"stage": "raw_dataset", "metric": "text_empty_non_null", "value": raw_text_empty},
    {"stage": "raw_dataset", "metric": "text_non_string_non_null", "value": raw_text_non_string},
    {"stage": "raw_dataset", "metric": "title_null", "value": raw_title_null if TITLE_COLUMN else None},
]
null_audit_rows.extend(
    {"stage": "raw_column_nulls", "metric": f"null__{column}", "value": count}
    for column, count in sorted(raw_column_nulls.items())
)
null_audit_rows.extend([
    {"stage": "eligibility", "metric": "eligible_rows_before_text_deduplication", "value": eligible_rows},
    {"stage": "eligibility", "metric": "duplicate_eligible_texts_by_hash64", "value": duplicate_eligible_texts},
    {"stage": "eligibility", "metric": "unique_eligible_rows", "value": len(unique_candidate_meta)},
    {"stage": "working_sample", "metric": "sampled_rows", "value": len(sample)},
    {"stage": "working_sample", "metric": "raw_text_null", "value": int(sample["raw_text"].isna().sum())},
    {"stage": "working_sample", "metric": "title_null", "value": int(sample["title"].isna().sum())},
    {"stage": "working_sample", "metric": "clean_text_null_before_filter", "value": int(sample["clean_text"].isna().sum())},
    {"stage": "post_preprocessing", "metric": "valid_rows", "value": len(valid_sample)},
    {"stage": "post_preprocessing", "metric": "clean_text_null", "value": int(valid_sample["clean_text"].isna().sum())},
    {"stage": "post_preprocessing", "metric": "token_count_null", "value": int(valid_sample["token_count"].isna().sum())},
    {"stage": "split", "metric": "train_rows", "value": len(train)},
    {"stage": "split", "metric": "topic_validation_rows", "value": len(topic_validation)},
    {"stage": "split", "metric": "link_test_rows", "value": len(test)},
    {"stage": "split", "metric": "unused_valid_rows", "value": len(unused)},
    {"stage": "split", "metric": "train_validation_overlap", "value": split_overlaps["train_validation"]},
    {"stage": "split", "metric": "train_link_test_overlap", "value": split_overlaps["train_link_test"]},
    {"stage": "split", "metric": "validation_link_test_overlap", "value": split_overlaps["validation_link_test"]},
])
null_audit = pd.DataFrame(null_audit_rows)

split_table = pd.DataFrame({
    "stratum": sorted(valid_sample["length_stratum"].astype(str).unique()),
})
for name, frame in (("valid_sample", valid_sample), ("train", train), ("topic_validation", topic_validation), ("link_test", test)):
    counts = frame["length_stratum"].astype(str).value_counts()
    split_table[name] = split_table["stratum"].map(counts).fillna(0).astype(int)

train_doc_ids = set(train["doc_id"].astype(int))
topic_validation_doc_ids = set(topic_validation["doc_id"].astype(int))
test_doc_ids = set(test["doc_id"].astype(int))
unused_doc_ids = set(unused["doc_id"].astype(int))
sample_provenance = sample[[
    "doc_id", "source_row", "title", "text_hash64_hex", "raw_chars", "persian_chars",
    "persian_ratio", "raw_length_stratum", "preprocessing_status"
]].copy()
sample_provenance["token_count"] = sample["clean_text"].str.split().str.len()
sample_provenance["length_stratum"] = sample["doc_id"].map(
    valid_sample.set_index("doc_id")["length_stratum"].astype(str)
)
sample_provenance["split"] = sample_provenance["doc_id"].map(
    lambda doc_id: (
        "train" if int(doc_id) in train_doc_ids else
        "topic_validation" if int(doc_id) in topic_validation_doc_ids else
        "link_test" if int(doc_id) in test_doc_ids else
        "unused_valid" if int(doc_id) in unused_doc_ids else
        "excluded_after_preprocessing"
    )
)
sample_provenance = sample_provenance.sort_values("source_row").reset_index(drop=True)

print("\nNull and provenance audit")
display(null_audit)
print("\nDuplicate-group distribution")
display(duplicate_group_distribution)
print("\nLargest exact-text hash groups")
display(top_duplicate_groups.head(20))
print("\nPreprocessing outcomes")
display(pd.DataFrame(sorted(preprocessing_reason_counts.items()), columns=["status", "documents"]))
print("\nLength-stratified split")
display(split_table)
print("\nToken-length summary")
display(valid_sample["token_count"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("token_count"))

# =============================================================================
# 5. COMMON TOPIC-MODEL EVALUATION
# =============================================================================

heading("2. Topic-model benchmark")
train_clean = train["clean_text"].astype(str).tolist()
topic_validation_clean = topic_validation["clean_text"].astype(str).tolist()
train_raw_context = [normalize_base_text(text, use_hazm=True) for text in train["raw_text"].astype(str)]
topic_validation_raw_context = [
    normalize_base_text(text, use_hazm=True) for text in topic_validation["raw_text"].astype(str)
]
train_tokens = [text.split() for text in train_clean]
topic_validation_tokens = [text.split() for text in topic_validation_clean]

# Shared training-derived vocabulary. All four core families use these exact
# lexical terms; contextual encoders remain model-specific by design.
shared_count_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=SHARED_VOCAB_MIN_DF,
    max_df=SHARED_VOCAB_MAX_DF,
    max_features=SHARED_VOCAB_SIZE,
    ngram_range=(1, 1),
)
shared_count_matrix = shared_count_vectorizer.fit_transform(train_clean)
shared_terms = np.asarray(shared_count_vectorizer.get_feature_names_out())
shared_vocabulary = {term: int(index) for index, term in enumerate(shared_terms)}
if not (100 <= len(shared_vocabulary) <= SHARED_VOCAB_SIZE):
    raise RuntimeError(f"Invalid shared vocabulary size: {len(shared_vocabulary)}")

term_document_frequency = np.asarray((shared_count_matrix > 0).sum(axis=0)).ravel().astype(int)
term_corpus_frequency = np.asarray(shared_count_matrix.sum(axis=0)).ravel().astype(int)
shared_vocabulary_table = pd.DataFrame({
    "term": shared_terms,
    "vocabulary_index": np.arange(len(shared_terms), dtype=int),
    "document_frequency": term_document_frequency,
    "corpus_frequency": term_corpus_frequency,
}).sort_values(["corpus_frequency", "term"], ascending=[False, True]).reset_index(drop=True)

# Preserve Gensim's real document/corpus counts while restricting LDA to the
# same terms used by NMF, CTM, and BERTopic.
train_dictionary = Dictionary(train_tokens)
bad_ids = [token_id for token, token_id in train_dictionary.token2id.items() if token not in shared_vocabulary]
train_dictionary.filter_tokens(bad_ids=bad_ids)
train_dictionary.compactify()
if set(train_dictionary.token2id) != set(shared_vocabulary):
    missing = set(shared_vocabulary) - set(train_dictionary.token2id)
    extra = set(train_dictionary.token2id) - set(shared_vocabulary)
    raise RuntimeError(f"Shared vocabulary mismatch: missing={len(missing)}, extra={len(extra)}")
train_corpus = [train_dictionary.doc2bow(tokens) for tokens in train_tokens]

# The complete topic-validation split is the sole reference for C_v/NPMI and
# CTM held-out topic distributions. Link-test documents are untouched here.
reference_texts = topic_validation_tokens[:COHERENCE_REFERENCE_DOCS]
reference_dictionary = Dictionary(reference_texts)
reference_bad_ids = [
    token_id for token, token_id in reference_dictionary.token2id.items() if token not in shared_vocabulary
]
reference_dictionary.filter_tokens(bad_ids=reference_bad_ids)
reference_dictionary.compactify()
if len(reference_dictionary) == 0:
    raise RuntimeError("Topic-validation reference vocabulary is empty after shared-vocabulary restriction.")

print(f"Shared training vocabulary: {len(shared_vocabulary):,}")
print(f"Held-out topic-validation documents: {len(reference_texts):,}")
print(f"Held-out reference vocabulary represented: {len(reference_dictionary):,}")
topic_data_summary = pd.DataFrame([
    {
        "training_documents": len(train),
        "topic_validation_documents": len(topic_validation),
        "link_test_documents_reserved": len(test),
        "shared_vocabulary_size": len(shared_vocabulary),
        "shared_vocabulary_min_df": SHARED_VOCAB_MIN_DF,
        "shared_vocabulary_max_df": SHARED_VOCAB_MAX_DF,
        "coherence_reference_documents": len(reference_texts),
        "coherence_reference_vocabulary": len(reference_dictionary),
        "top_n_words": TOP_N_WORDS,
        "candidate_words": TOPIC_CANDIDATE_WORDS,
    }
])
display(topic_data_summary)
print("\nMost frequent terms in the shared training vocabulary")
display(shared_vocabulary_table.head(30))
del shared_count_matrix
gc.collect()


def canonicalize_topic_lists(raw_topics: Sequence[Sequence[str]]) -> list[list[str]]:
    canonical_topics: list[list[str]] = []
    for topic in raw_topics:
        words: list[str] = []
        seen: set[str] = set()
        for raw_word in list(topic)[:TOPIC_CANDIDATE_WORDS]:
            word = canonical_topic_term(raw_word)
            if not word or word in seen:
                continue
            seen.add(word)
            words.append(word)
            if len(words) == TOP_N_WORDS:
                break
        if words:
            canonical_topics.append(words)
    return canonical_topics


def evaluate_topics_against_reference(
    raw_topics: Sequence[Sequence[str]],
    evaluation_texts: Sequence[Sequence[str]],
    evaluation_dictionary: Dictionary,
) -> tuple[dict[str, float], list[list[str]]]:
    """Evaluate topic lists consistently while separating lexical and corpus metrics.

    Diversity, distinctness, and stability use canonical top-word lists directly.
    C_v and NPMI use only words available in the held-out reference dictionary.
    """
    canonical_topics = canonicalize_topic_lists(raw_topics)
    matched_topics: list[list[str]] = []
    matched_word_counts: list[int] = []
    for topic in canonical_topics:
        matched = [word for word in topic if word in evaluation_dictionary.token2id][:TOP_N_WORDS]
        matched_word_counts.append(len(matched))
        if len(matched) >= 3:
            matched_topics.append(matched)

    valid_fraction = len(matched_topics) / len(canonical_topics) if canonical_topics else float("nan")
    matched_terms = sum(matched_word_counts)
    possible_terms = TOP_N_WORDS * len(canonical_topics)
    diagnostics = {
        "raw_topic_count": len(raw_topics),
        "canonical_topic_count": len(canonical_topics),
        "valid_topic_count": len(matched_topics),
        "valid_topic_fraction": valid_fraction,
        "mean_matched_words_per_topic": float(np.mean(matched_word_counts)) if matched_word_counts else float("nan"),
        "reference_match_rate": matched_terms / possible_terms if possible_terms else float("nan"),
    }

    if len(matched_topics) < 2:
        return {
            "c_v": float("nan"),
            "c_npmi": float("nan"),
            "topic_coherence_sd": float("nan"),
            "diversity": topic_diversity(canonical_topics),
            "distinctness": topic_distinctness(canonical_topics),
            **diagnostics,
        }, canonical_topics

    cv_model = CoherenceModel(
        topics=matched_topics,
        texts=evaluation_texts,
        dictionary=evaluation_dictionary,
        coherence="c_v",
        topn=TOP_N_WORDS,
        processes=1,
    )
    cv_per_topic = np.asarray(cv_model.get_coherence_per_topic(), dtype=float)
    npmi = CoherenceModel(
        topics=matched_topics,
        texts=evaluation_texts,
        dictionary=evaluation_dictionary,
        coherence="c_npmi",
        topn=TOP_N_WORDS,
        processes=1,
    ).get_coherence()
    return {
        "c_v": float(np.nanmean(cv_per_topic)),
        "c_npmi": float(npmi),
        "topic_coherence_sd": float(np.nanstd(cv_per_topic, ddof=1)) if len(cv_per_topic) > 1 else 0.0,
        "diversity": topic_diversity(canonical_topics),
        "distinctness": topic_distinctness(canonical_topics),
        **diagnostics,
    }, canonical_topics


def topic_diversity(topics: Sequence[Sequence[str]]) -> float:
    words = [word for topic in topics for word in list(topic)[:TOP_N_WORDS]]
    return len(set(words)) / len(words) if words else float("nan")


def topic_distinctness(topics: Sequence[Sequence[str]]) -> float:
    sets = [set(topic[:TOP_N_WORDS]) for topic in topics if topic]
    if len(sets) < 2:
        return float("nan")
    similarities = []
    for left, right in combinations(sets, 2):
        union = left | right
        similarities.append(len(left & right) / len(union) if union else 0.0)
    return 1.0 - float(np.mean(similarities))


def evaluate_topics_common(raw_topics: Sequence[Sequence[str]]) -> tuple[dict[str, float], list[list[str]]]:
    return evaluate_topics_against_reference(raw_topics, reference_texts, reference_dictionary)


def topic_alignment_stability(seed_topics: Sequence[Sequence[Sequence[str]]]) -> float:
    if len(seed_topics) < 2:
        return float("nan")
    pair_scores: list[float] = []
    for topics_a, topics_b in combinations(seed_topics, 2):
        sets_a = [set(topic[:TOP_N_WORDS]) for topic in topics_a if topic]
        sets_b = [set(topic[:TOP_N_WORDS]) for topic in topics_b if topic]
        if not sets_a or not sets_b:
            continue
        similarity = np.zeros((len(sets_a), len(sets_b)), dtype=float)
        for i, left in enumerate(sets_a):
            for j, right in enumerate(sets_b):
                union = left | right
                similarity[i, j] = len(left & right) / len(union) if union else 0.0
        rows, cols = linear_sum_assignment(-similarity)
        pair_scores.append(float(similarity[rows, cols].mean()))
    return float(np.mean(pair_scores)) if pair_scores else float("nan")


def run_record(
    model: str,
    k: int,
    seed: int,
    metrics: dict[str, float],
    seconds: float,
    docs: int,
    *,
    model_family: str | None = None,
    benchmark_role: str = "core",
    encoder: str | None = None,
    **extra: Any,
) -> dict[str, Any]:
    return {
        "model": model,
        "model_family": model_family or model,
        "benchmark_role": benchmark_role,
        "encoder": encoder,
        "k": int(k),
        "seed": int(seed),
        **metrics,
        "training_seconds": float(seconds),
        "training_documents": int(docs),
        **extra,
    }


all_run_rows: list[dict[str, Any]] = []
variant_topics: dict[str, dict[tuple[int, int], list[list[str]]]] = {
    "LDA": {},
    "NMF": {},
    "CTM": {},
    "BERTopic": {},
    "BERTopic-ParsBERT": {},
}
selected_lda_models: dict[int, LdaModel] = {}

# ----- LDA -----
for seed in SEEDS:
    for k in tqdm(TOPIC_COUNTS, desc=f"LDA seed={seed}"):
        set_seed(seed)
        started = time.time()
        model = LdaModel(
            corpus=train_corpus,
            id2word=train_dictionary,
            num_topics=k,
            random_state=seed,
            passes=LDA_PASSES,
            iterations=LDA_ITERATIONS,
            alpha=LDA_ALPHA,
            eta=LDA_ETA,
            minimum_probability=0.0,
            eval_every=None,
        )
        raw_topics = [
            [word for word, _weight in model.show_topic(topic_id, topn=TOPIC_CANDIDATE_WORDS)]
            for topic_id in range(k)
        ]
        metrics, canonical_topics = evaluate_topics_common(raw_topics)
        all_run_rows.append(run_record(
            "LDA", k, seed, metrics, time.time() - started, len(train),
            model_family="LDA", benchmark_role="core",
            shared_vocabulary_size=len(shared_vocabulary),
        ))
        variant_topics["LDA"][(k, seed)] = canonical_topics
        if seed == GRAPH_TOPIC_SEED and k in GRAPH_TOPIC_K_SENSITIVITY:
            selected_lda_models[k] = model
        else:
            del model

# ----- NMF -----
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    vocabulary=shared_vocabulary,
    sublinear_tf=True,
)
tfidf_matrix = tfidf_vectorizer.fit_transform(train_clean)
nmf_features = np.asarray(tfidf_vectorizer.get_feature_names_out())
for seed in SEEDS:
    for k in tqdm(TOPIC_COUNTS, desc=f"NMF seed={seed}"):
        set_seed(seed)
        started = time.time()
        model = NMF(
            n_components=k,
            init=NMF_INIT,
            solver="mu",
            beta_loss="frobenius",
            max_iter=NMF_MAX_ITER,
            tol=NMF_TOL,
            alpha_W=NMF_ALPHA_W,
            alpha_H=NMF_ALPHA_H,
            l1_ratio=NMF_L1_RATIO,
            random_state=seed,
        )
        model.fit(tfidf_matrix)
        raw_topics = [
            nmf_features[np.argsort(component)[::-1][:TOPIC_CANDIDATE_WORDS]].tolist()
            for component in model.components_
        ]
        metrics, canonical_topics = evaluate_topics_common(raw_topics)
        all_run_rows.append(run_record(
            "NMF", k, seed, metrics, time.time() - started, len(train),
            model_family="NMF", benchmark_role="core",
            converged=bool(model.n_iter_ < NMF_MAX_ITER), n_iter=int(model.n_iter_),
            shared_vocabulary_size=len(shared_vocabulary),
        ))
        variant_topics["NMF"][(k, seed)] = canonical_topics
        del model

del tfidf_matrix
gc.collect()


def encode_contextual_documents(
    documents: Sequence[str],
    model_name: str,
    max_seq_length: int,
    batch_size: int,
    description: str,
) -> np.ndarray:
    log(f"Encoding {len(documents):,} documents with {description}: {model_name}")
    encoder = build_sentence_encoder(
        model_name, max_seq_length=max_seq_length, device=DEVICE, revision=MODEL_REVISIONS[model_name]
    )
    embeddings = encoder.encode(
        list(documents),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=False,
    ).astype(np.float32, copy=False)
    del encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if embeddings.shape[0] != len(documents) or not np.isfinite(embeddings).all():
        raise RuntimeError(f"Invalid embeddings from {model_name}: shape={embeddings.shape}")
    return embeddings


def run_bertopic_variant(
    *,
    model_label: str,
    encoder_label: str,
    encoder_model_id: str,
    embeddings: np.ndarray,
    benchmark_role: str,
) -> None:
    """Embedding -> UMAP -> KMeans labels -> official BERTopic c-TF-IDF.

    BERTopic's documented manual-topic interface is used: BaseEmbedder,
    BaseDimensionalityReduction and BaseCluster skip internal embedding/reduction/
    clustering, while externally generated KMeans labels are passed through ``y``.
    Persian lexical input is already whitespace-tokenized; therefore the vectorizer
    uses ``str.split`` and topic-document thresholds min_df=1/max_df=1.0.
    """
    for seed in SEEDS:
        set_seed(seed)
        umap_model = UMAP(
            n_neighbors=BERTOPIC_UMAP_NEIGHBORS,
            n_components=BERTOPIC_UMAP_COMPONENTS,
            min_dist=BERTOPIC_UMAP_MIN_DIST,
            metric="cosine",
            random_state=seed,
            low_memory=True,
        )
        reduced_embeddings = umap_model.fit_transform(embeddings)
        if reduced_embeddings.shape != (len(train_clean), BERTOPIC_UMAP_COMPONENTS):
            raise RuntimeError(f"Unexpected UMAP shape for {model_label}: {reduced_embeddings.shape}")
        if not np.isfinite(reduced_embeddings).all():
            raise RuntimeError(f"UMAP produced non-finite values for {model_label}, seed={seed}")

        for k in tqdm(TOPIC_COUNTS, desc=f"{model_label} seed={seed}"):
            started = time.time()
            cluster_model = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=BERTOPIC_KMEANS_N_INIT,
                max_iter=BERTOPIC_KMEANS_MAX_ITER,
                algorithm="lloyd",
            )
            labels = cluster_model.fit_predict(reduced_embeddings).astype(int)
            unique_labels, label_counts = np.unique(labels, return_counts=True)
            if len(unique_labels) != k or set(unique_labels.tolist()) != set(range(k)):
                raise RuntimeError(
                    f"{model_label} KMeans returned {len(unique_labels)} non-empty clusters for K={k}; "
                    f"labels={unique_labels[:20].tolist()}"
                )

            # Diagnose the exact topic-level corpus before BERTopic sees it.
            grouped = (
                pd.DataFrame({"Document": train_clean, "Topic": labels})
                .groupby("Topic", as_index=False, sort=True)["Document"]
                .agg(" ".join)
            )
            if len(grouped) != k or grouped["Document"].str.strip().eq("").any():
                raise RuntimeError(f"Invalid grouped topic corpus for {model_label}, K={k}, seed={seed}")

            vectorizer_model = CountVectorizer(
                tokenizer=str.split,
                preprocessor=None,
                token_pattern=None,
                lowercase=False,
                vocabulary=shared_vocabulary,
                ngram_range=(1, 1),
            )
            diagnostic_matrix = vectorizer_model.fit_transform(grouped["Document"])
            empty_group_rows = int((diagnostic_matrix.getnnz(axis=1) == 0).sum())
            diagnostic_vocabulary = int(diagnostic_matrix.shape[1])
            if diagnostic_vocabulary == 0 or empty_group_rows:
                raise RuntimeError(
                    f"Topic-level Persian vectorization failed before BERTopic: model={model_label}, "
                    f"K={k}, seed={seed}, vocabulary={diagnostic_vocabulary}, empty_rows={empty_group_rows}"
                )

            # Recreate the vectorizer so BERTopic fits a fresh, auditable vocabulary.
            vectorizer_model = CountVectorizer(
                tokenizer=str.split,
                preprocessor=None,
                token_pattern=None,
                lowercase=False,
                vocabulary=shared_vocabulary,
                ngram_range=(1, 1),
            )
            topic_model = BERTopic(
                embedding_model=BaseEmbedder(),
                umap_model=BaseDimensionalityReduction(),
                hdbscan_model=BaseCluster(),
                vectorizer_model=vectorizer_model,
                ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
                top_n_words=TOPIC_CANDIDATE_WORDS,
                calculate_probabilities=False,
                low_memory=True,
                verbose=False,
            )
            assigned_topics, _ = topic_model.fit_transform(train_clean, y=labels)
            if len(assigned_topics) != len(train_clean):
                raise RuntimeError("BERTopic did not return one assigned topic per document.")
            if topic_model.c_tf_idf_ is None or topic_model.c_tf_idf_.shape[0] != k:
                raise RuntimeError(
                    f"Invalid official c-TF-IDF matrix for {model_label}, K={k}: "
                    f"{None if topic_model.c_tf_idf_ is None else topic_model.c_tf_idf_.shape}"
                )

            topic_ids = sorted(int(topic_id) for topic_id in topic_model.get_topics() if int(topic_id) >= 0)
            raw_topics: list[list[str]] = []
            for topic_id in topic_ids:
                representation = topic_model.get_topic(topic_id) or []
                raw_topics.append([
                    word for word, weight in representation[:TOPIC_CANDIDATE_WORDS]
                    if word and np.isfinite(weight)
                ])
            if len(raw_topics) != k:
                raise RuntimeError(f"{model_label} represented {len(raw_topics)} topics for requested K={k}.")

            metrics, canonical_topics = evaluate_topics_common(raw_topics)
            if metrics["valid_topic_fraction"] < 0.95 or not np.isfinite(
                [metrics["c_v"], metrics["c_npmi"], metrics["diversity"], metrics["distinctness"]]
            ).all():
                raise RuntimeError(
                    f"{model_label} evaluation failed for K={k}, seed={seed}: {metrics}; "
                    f"sample_topic={raw_topics[0][:10] if raw_topics else []}"
                )

            all_run_rows.append(run_record(
                model_label,
                k,
                seed,
                metrics,
                time.time() - started,
                len(train),
                model_family="BERTopic",
                benchmark_role=benchmark_role,
                encoder=encoder_label,
                encoder_model_id=encoder_model_id,
                actual_topics=len(raw_topics),
                minimum_cluster_size=int(label_counts.min()),
                maximum_cluster_size=int(label_counts.max()),
                topic_level_vocabulary=diagnostic_vocabulary,
                empty_topic_documents_before_fit=empty_group_rows,
                c_tfidf_used=True,
                ctfidf_implementation="BERTopic ClassTfidfTransformer (official manual-label workflow)",
                cluster_labels="external UMAP+KMeans labels passed via y",
                lexical_input="clean Persian whitespace-tokenized unigrams",
                contextual_input="normalized raw Persian text",
                shared_vocabulary_size=len(shared_vocabulary),
            ))
            variant_topics[model_label][(k, seed)] = canonical_topics
            del topic_model, cluster_model, diagnostic_matrix, grouped
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del reduced_embeddings, umap_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ----- multilingual contextual embeddings: core BERTopic + full-data CTM -----
multilingual_train_embeddings = encode_contextual_documents(
    train_raw_context,
    MULTILINGUAL_ENCODER_MODEL,
    MULTILINGUAL_MAX_SEQ_LENGTH,
    EMBEDDING_BATCH_SIZE,
    "multilingual core encoder (train)",
)
multilingual_validation_embeddings = encode_contextual_documents(
    topic_validation_raw_context,
    MULTILINGUAL_ENCODER_MODEL,
    MULTILINGUAL_MAX_SEQ_LENGTH,
    EMBEDDING_BATCH_SIZE,
    "multilingual core encoder (topic validation)",
)
run_bertopic_variant(
    model_label="BERTopic",
    encoder_label="multilingual-MiniLM",
    encoder_model_id=MULTILINGUAL_ENCODER_MODEL,
    embeddings=multilingual_train_embeddings,
    benchmark_role="core",
)

# ----- CTM: official CombinedTM training loop -----
ctm_train = train.iloc[:CTM_TRAIN_DOCS].reset_index(drop=True)
ctm_test = topic_validation.iloc[:CTM_TEST_DOCS].reset_index(drop=True)
if len(ctm_train) != CTM_TRAIN_DOCS or len(ctm_test) != CTM_TEST_DOCS:
    raise RuntimeError(f"CTM requires exactly {CTM_TRAIN_DOCS:,} training and {CTM_TEST_DOCS:,} topic-validation documents in this protocol.")

ctm_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    vocabulary=shared_vocabulary,
)
ctm_train_bow = ctm_vectorizer.fit_transform(ctm_train["clean_text"].astype(str))
ctm_test_bow = ctm_vectorizer.transform(ctm_test["clean_text"].astype(str))
ctm_vocab = ctm_vectorizer.get_feature_names_out()
ctm_id2token = {index: token for index, token in enumerate(ctm_vocab)}
ctm_training_dataset = CTMDataset(
    X_contextual=multilingual_train_embeddings[:CTM_TRAIN_DOCS],
    X_bow=ctm_train_bow,
    idx2token=ctm_id2token,
)
ctm_testing_dataset = CTMDataset(
    X_contextual=multilingual_validation_embeddings[:CTM_TEST_DOCS],
    X_bow=ctm_test_bow,
    idx2token=ctm_id2token,
)
ctm_bow_size = len(ctm_vocab)
ctm_contextual_size = int(multilingual_train_embeddings.shape[1])
if ctm_bow_size == 0 or ctm_bow_size > CTM_MAX_FEATURES:
    raise RuntimeError(f"Invalid CTM vocabulary size: {ctm_bow_size}")

for seed in SEEDS:
    for k in tqdm(TOPIC_COUNTS, desc=f"CTM seed={seed}"):
        set_seed(seed)
        started = time.time()
        kwargs = dict(
            bow_size=ctm_bow_size,
            contextual_size=ctm_contextual_size,
            n_components=k,
            num_epochs=CTM_EPOCHS,
            batch_size=CTM_BATCH_SIZE,
            num_data_loader_workers=0,
            learn_priors=True,
        )
        if "lr" in inspect.signature(CombinedTM).parameters:
            kwargs["lr"] = CTM_LEARNING_RATE
        model = CombinedTM(**kwargs)
        model.fit(ctm_training_dataset, verbose=False, do_train_predictions=False)
        raw_topics = model.get_topic_lists(TOPIC_CANDIDATE_WORDS)
        metrics, canonical_topics = evaluate_topics_common(raw_topics)
        if metrics["valid_topic_fraction"] < 0.95 or not np.isfinite(
            [metrics["c_v"], metrics["c_npmi"], metrics["diversity"], metrics["distinctness"]]
        ).all():
            raise RuntimeError(f"CTM produced invalid topic evidence for K={k}, seed={seed}: {metrics}")
        distribution = model.get_doc_topic_distribution(
            ctm_testing_dataset,
            n_samples=CTM_TEST_TOPIC_SAMPLES,
        )
        inferred_shape = list(distribution.shape)
        if inferred_shape != [CTM_TEST_DOCS, k] or not np.isfinite(distribution).all():
            raise RuntimeError(f"Invalid CTM topic-validation distribution for K={k}: shape={inferred_shape}")
        all_run_rows.append(run_record(
            "CTM",
            k,
            seed,
            metrics,
            time.time() - started,
            len(ctm_train),
            model_family="CTM",
            benchmark_role="core",
            encoder="multilingual-MiniLM",
            encoder_model_id=MULTILINGUAL_ENCODER_MODEL,
            topic_validation_documents_reserved=len(ctm_test),
            topic_validation_topic_distribution_shape=inferred_shape,
            epochs=CTM_EPOCHS,
            batch_size=CTM_BATCH_SIZE,
            controlled_bow_vocabulary=ctm_bow_size,
            shared_vocabulary_size=len(shared_vocabulary),
            training_implementation="official CombinedTM.fit",
        ))
        variant_topics["CTM"][(k, seed)] = canonical_topics
        del model, distribution
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Multilingual embeddings are no longer needed after core BERTopic and CTM.
del multilingual_train_embeddings, multilingual_validation_embeddings, ctm_train_bow, ctm_test_bow
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----- ParsBERT BERTopic encoder sensitivity -----
# FIXED: ParsBERT encoding now works with the updated build_sentence_encoder function
parsbert_train_embeddings = encode_contextual_documents(
    train_raw_context,
    PARSBERT_ENCODER_MODEL,
    PARSBERT_MAX_SEQ_LENGTH,
    PARSBERT_BATCH_SIZE,
    "ParsBERT mean-pooled encoder sensitivity",
)
run_bertopic_variant(
    model_label="BERTopic-ParsBERT",
    encoder_label="ParsBERT-mean-pooling",
    encoder_model_id=PARSBERT_ENCODER_MODEL,
    embeddings=parsbert_train_embeddings,
    benchmark_role="encoder_sensitivity",
)
del parsbert_train_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----- run validation and aggregation -----
all_runs = pd.DataFrame(all_run_rows).sort_values(["benchmark_role", "model", "k", "seed"]).reset_index(drop=True)
core_runs = all_runs[all_runs["benchmark_role"] == "core"].copy()
bertopic_encoder_runs = all_runs[all_runs["model_family"] == "BERTopic"].copy()

expected_core_runs = 4 * len(TOPIC_COUNTS) * len(SEEDS)
expected_parsbert_extra_runs = len(TOPIC_COUNTS) * len(SEEDS)
expected_total_runs = expected_core_runs + expected_parsbert_extra_runs
if len(core_runs) != expected_core_runs:
    raise AssertionError(f"Expected {expected_core_runs} core topic runs, obtained {len(core_runs)}.")
if len(all_runs) != expected_total_runs:
    raise AssertionError(f"Expected {expected_total_runs} total topic runs, obtained {len(all_runs)}.")

core_metric_columns = ["c_v", "c_npmi", "diversity", "distinctness"]
if not np.isfinite(all_runs[core_metric_columns].to_numpy(dtype=float)).all():
    bad = all_runs.loc[~np.isfinite(all_runs[core_metric_columns].to_numpy(dtype=float)).all(axis=1)]
    raise RuntimeError(f"Non-finite topic metrics detected:\n{bad}")

aggregate_rows: list[dict[str, Any]] = []
for (model, k), group in all_runs.groupby(["model", "k"], sort=True):
    row: dict[str, Any] = {
        "model": model,
        "model_family": group["model_family"].iloc[0],
        "benchmark_role": group["benchmark_role"].iloc[0],
        "encoder": group["encoder"].iloc[0],
        "k": int(k),
        "seeds": len(group),
        "valid_topic_fraction_mean": group["valid_topic_fraction"].mean(),
        "reference_match_rate_mean": group["reference_match_rate"].mean(),
        "training_seconds_mean": group["training_seconds"].mean(),
        "training_seconds_sd": group["training_seconds"].std(ddof=1),
    }
    for metric_index, metric in enumerate(core_metric_columns):
        ci_low, ci_high = bootstrap_mean_ci(
            group[metric],
            seed=RANDOM_SEED + 1000 * metric_index + int(k) + len(aggregate_rows),
        )
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_sd"] = group[metric].std(ddof=1)
        row[f"{metric}_bootstrap_ci_low"] = ci_low
        row[f"{metric}_bootstrap_ci_high"] = ci_high
    aggregate_rows.append(row)
aggregate = pd.DataFrame(aggregate_rows)

stability_rows: list[dict[str, Any]] = []
for model_name, model_topics in variant_topics.items():
    for k in TOPIC_COUNTS:
        seed_topics = [model_topics[(k, seed)] for seed in SEEDS]
        stability_rows.append({
            "model": model_name,
            "k": k,
            "stability": topic_alignment_stability(seed_topics),
        })
aggregate = aggregate.merge(pd.DataFrame(stability_rows), on=["model", "k"], how="left")
core_aggregate = aggregate[aggregate["benchmark_role"] == "core"].copy()
bertopic_encoder_aggregate = aggregate[aggregate["model_family"] == "BERTopic"].copy()

# Primary comparison at one common topic count prevents model-specific K selection
# from being confused with a matched model comparison.
common_k_results = (
    core_aggregate[core_aggregate["k"] == PRIMARY_COMMON_K]
    .sort_values("c_v_mean", ascending=False)
    .reset_index(drop=True)
)
if set(common_k_results["model"]) != {"LDA", "NMF", "CTM", "BERTopic"}:
    raise RuntimeError(f"Incomplete common-K comparison at K={PRIMARY_COMMON_K}")

metric_optima_rows: list[dict[str, Any]] = []
for model_name, group in aggregate.groupby("model", sort=True):
    for metric in ("c_v_mean", "c_npmi_mean", "diversity_mean", "distinctness_mean", "stability"):
        best_row = group.sort_values([metric, "k"], ascending=[False, True]).iloc[0]
        metric_optima_rows.append({
            "model": model_name,
            "benchmark_role": best_row["benchmark_role"],
            "encoder": best_row["encoder"],
            "metric": metric,
            "k": int(best_row["k"]),
            "value": float(best_row[metric]),
        })
metric_optima = pd.DataFrame(metric_optima_rows)


def pareto_mask(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
    values = frame[list(columns)].to_numpy(dtype=float)
    efficient = np.ones(len(values), dtype=bool)
    for i, candidate in enumerate(values):
        if not efficient[i]:
            continue
        dominates_candidate = np.all(values >= candidate, axis=1) & np.any(values > candidate, axis=1)
        dominates_candidate[i] = False
        if dominates_candidate.any():
            efficient[i] = False
    return efficient

pareto_columns = ["c_v_mean", "c_npmi_mean", "diversity_mean", "distinctness_mean", "stability"]
core_pareto_front = core_aggregate.loc[pareto_mask(core_aggregate, pareto_columns)].copy()
core_pareto_front = core_pareto_front.sort_values(["model", "k"]).reset_index(drop=True)

best_configs = (
    aggregate.sort_values(["model", "c_v_mean", "diversity_mean"], ascending=[True, False, False])
    .groupby("model", as_index=False)
    .head(1)
    .sort_values("c_v_mean", ascending=False)
    .reset_index(drop=True)
)
core_best_configs = best_configs[best_configs["benchmark_role"] == "core"].reset_index(drop=True)

representative_topic_rows: list[dict[str, Any]] = []
for row in best_configs.itertuples(index=False):
    topics = variant_topics[row.model][(int(row.k), RANDOM_SEED)]
    for topic_index, words in enumerate(topics[:min(20, len(topics))]):
        representative_topic_rows.append({
            "model": row.model,
            "encoder": row.encoder,
            "k": int(row.k),
            "seed": RANDOM_SEED,
            "topic_index": topic_index,
            "top_words": " | ".join(words[:TOP_N_WORDS]),
        })
representative_topics = pd.DataFrame(representative_topic_rows)

run_metadata_lookup = all_runs.set_index(["model", "k", "seed"])[
    ["model_family", "benchmark_role", "encoder"]
].to_dict(orient="index")
all_topic_word_rows: list[dict[str, Any]] = []
for model_name, run_topics in variant_topics.items():
    for (k, seed), topics in sorted(run_topics.items()):
        metadata = run_metadata_lookup[(model_name, int(k), int(seed))]
        for topic_index, words in enumerate(topics):
            all_topic_word_rows.append({
                "model": model_name,
                "model_family": metadata["model_family"],
                "benchmark_role": metadata["benchmark_role"],
                "encoder": metadata["encoder"],
                "k": int(k),
                "seed": int(seed),
                "topic_index": int(topic_index),
                "word_count": len(words),
                "top_words": " | ".join(words[:TOP_N_WORDS]),
            })
all_topic_words = pd.DataFrame(all_topic_word_rows).sort_values(
    ["benchmark_role", "model", "k", "seed", "topic_index"]
).reset_index(drop=True)


def make_topic_audit_template(topics_per_model: int = 10) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for model_name in ["LDA", "NMF", "CTM", "BERTopic", "BERTopic-ParsBERT"]:
        topics = variant_topics[model_name][(PRIMARY_COMMON_K, RANDOM_SEED)]
        chosen = np.linspace(0, len(topics) - 1, num=min(topics_per_model, len(topics)), dtype=int)
        for topic_index in sorted(set(chosen.tolist())):
            rows.append({
                "model": model_name,
                "k": PRIMARY_COMMON_K,
                "seed": RANDOM_SEED,
                "topic_index": int(topic_index),
                "top_words": " | ".join(topics[topic_index][:TOP_N_WORDS]),
                "proposed_label": "",
                "semantic_coherence_1_to_5": "",
                "intruder_word_if_any": "",
                "annotator": "",
                "notes": "",
            })
    return pd.DataFrame(rows)


topic_audit_template = make_topic_audit_template()

print(f"\nAll topic-model runs: {expected_core_runs} core runs + {expected_parsbert_extra_runs} ParsBERT sensitivity runs")
if SHOW_ALL_RUN_ROWS:
    display(all_runs)
print("\nCore four-model run-level results")
display(core_runs)
print("\nSeed-aggregated topic-model results with 5,000-resample bootstrap intervals")
display(aggregate)
print(f"\nPrimary matched comparison at common K={PRIMARY_COMMON_K}")
display(common_k_results)
print("\nMetric-specific optima (no composite quality score)")
display(metric_optima)
print("\nPareto-efficient core configurations across C_v, NPMI, diversity, distinctness, and stability")
display(core_pareto_front)
print("\nCoherence-maximizing configuration for each model/encoder variant")
display(best_configs)
print("\nRepresentative topics from each coherence-maximizing configuration")
display(representative_topics)
print(f"\nComplete topic-word evidence contains {len(all_topic_words):,} topic rows and is saved in the final JSON evidence.")
display(all_topic_words.head(200))
print("\nUnscored topic human-audit template (complete template saved in JSON)")
display(topic_audit_template)

# =============================================================================
# 6. METRIC-SPECIFIC CONFIGURATION-LEVEL STATISTICS
# =============================================================================

heading("3. Statistical testing")

topic_friedman_rows: list[dict[str, Any]] = []
topic_pairwise_rows: list[dict[str, Any]] = []
core_model_order = ["LDA", "NMF", "CTM", "BERTopic"]
run_level_metrics = ["c_v", "c_npmi", "diversity", "distinctness"]

statistical_core_runs = core_runs[core_runs["k"].isin(STATISTICAL_TOPIC_COUNTS)].copy()
statistical_encoder_runs = bertopic_encoder_runs[bertopic_encoder_runs["k"].isin(STATISTICAL_TOPIC_COUNTS)].copy()
expected_statistical_blocks = len(STATISTICAL_TOPIC_COUNTS) * len(SEEDS)

# Tests use the original evenly spaced K grid; added low-K values remain
# descriptive sensitivity points and cannot change the weighting of inference.
for metric in run_level_metrics:
    pivot = (
        statistical_core_runs.pivot_table(index=["k", "seed"], columns="model", values=metric, aggfunc="first")
        .reindex(columns=core_model_order)
        .dropna()
    )
    if len(pivot) != expected_statistical_blocks:
        raise RuntimeError(f"Incomplete paired blocks for topic metric {metric}: {len(pivot)}")
    statistic, p_value = friedmanchisquare(*[pivot[model].to_numpy(dtype=float) for model in core_model_order])
    topic_friedman_rows.append({
        "analysis": "core_topic_models",
        "metric": metric,
        "block_unit": "K × seed",
        "blocks": len(pivot),
        "models": ", ".join(core_model_order),
        "friedman_chi_square": float(statistic),
        "raw_p": float(p_value),
        "kendalls_w": float(statistic / (len(pivot) * (len(core_model_order) - 1))),
        "alpha": ALPHA,
        "significant_0_05": bool(p_value < ALPHA),
        "inference_scope": "Exploratory repeated-configuration comparison; K and seed blocks are not population samples.",
    })

    temporary: list[dict[str, Any]] = []
    pivot_reset = pivot.reset_index()
    for pair_index, (left, right) in enumerate(combinations(core_model_order, 2)):
        differences = pivot_reset[left] - pivot_reset[right]
        try:
            w_stat, pair_p = wilcoxon(
                pivot_reset[left],
                pivot_reset[right],
                zero_method="wilcox",
                alternative="two-sided",
                method="auto",
            )
        except ValueError:
            w_stat, pair_p = 0.0, 1.0
        difference_frame = pivot_reset[["k", "seed"]].copy()
        difference_frame["difference"] = differences.to_numpy(dtype=float)
        ci_low, ci_high = cluster_bootstrap_mean_ci(
            difference_frame,
            value_column="difference",
            cluster_column="k",
            seed=RANDOM_SEED + 10_000 * run_level_metrics.index(metric) + pair_index,
        )
        temporary.append({
            "analysis": "core_topic_models",
            "metric": metric,
            "model_a": left,
            "model_b": right,
            "paired_blocks": len(differences),
            "mean_difference_a_minus_b": float(differences.mean()),
            "cluster_bootstrap_95_ci_low": ci_low,
            "cluster_bootstrap_95_ci_high": ci_high,
            "bootstrap_clusters": "K",
            "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
            "wilcoxon_statistic": float(w_stat),
            "raw_p": float(pair_p),
            "rank_biserial": rank_biserial_from_differences(differences),
            "alpha": ALPHA,
            "inference_scope": "Exploratory; paired K × seed results with K-cluster bootstrap.",
        })
    adjusted = holm_adjust([row["raw_p"] for row in temporary])
    for row, adjusted_p in zip(temporary, adjusted):
        row["holm_p"] = adjusted_p
        row["significant_holm_0_05"] = bool(adjusted_p < ALPHA)
        topic_pairwise_rows.append(row)

# Stability is defined once per K after three-seed alignment, so its block is K.
stability_pivot = (
    core_aggregate[core_aggregate["k"].isin(STATISTICAL_TOPIC_COUNTS)].pivot(index="k", columns="model", values="stability")
    .reindex(columns=core_model_order)
    .dropna()
)
if len(stability_pivot) == len(STATISTICAL_TOPIC_COUNTS):
    statistic, p_value = friedmanchisquare(*[stability_pivot[model].to_numpy(dtype=float) for model in core_model_order])
    topic_friedman_rows.append({
        "analysis": "core_topic_models",
        "metric": "stability",
        "block_unit": "K",
        "blocks": len(stability_pivot),
        "models": ", ".join(core_model_order),
        "friedman_chi_square": float(statistic),
        "raw_p": float(p_value),
        "kendalls_w": float(statistic / (len(stability_pivot) * (len(core_model_order) - 1))),
        "alpha": ALPHA,
        "significant_0_05": bool(p_value < ALPHA),
        "inference_scope": f"Exploratory stability comparison across {len(STATISTICAL_TOPIC_COUNTS)} predeclared statistical K values.",
    })
    temporary = []
    stability_reset = stability_pivot.reset_index()
    for pair_index, (left, right) in enumerate(combinations(core_model_order, 2)):
        differences = stability_reset[left] - stability_reset[right]
        try:
            w_stat, pair_p = wilcoxon(stability_reset[left], stability_reset[right], method="auto")
        except ValueError:
            w_stat, pair_p = 0.0, 1.0
        ci_low, ci_high = bootstrap_mean_ci(
            differences,
            seed=RANDOM_SEED + 90_000 + pair_index,
        )
        temporary.append({
            "analysis": "core_topic_models",
            "metric": "stability",
            "model_a": left,
            "model_b": right,
            "paired_blocks": len(differences),
            "mean_difference_a_minus_b": float(differences.mean()),
            "cluster_bootstrap_95_ci_low": ci_low,
            "cluster_bootstrap_95_ci_high": ci_high,
            "bootstrap_clusters": "K values",
            "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
            "wilcoxon_statistic": float(w_stat),
            "raw_p": float(pair_p),
            "rank_biserial": rank_biserial_from_differences(differences),
            "alpha": ALPHA,
            "inference_scope": "Exploratory; only nine K-level stability observations.",
        })
    adjusted = holm_adjust([row["raw_p"] for row in temporary])
    for row, adjusted_p in zip(temporary, adjusted):
        row["holm_p"] = adjusted_p
        row["significant_holm_0_05"] = bool(adjusted_p < ALPHA)
        topic_pairwise_rows.append(row)

friedman_results = pd.DataFrame(topic_friedman_rows)
pairwise_topic_results = pd.DataFrame(topic_pairwise_rows)

# Predeclared BERTopic encoder sensitivity: multilingual MiniLM vs mean-pooled ParsBERT.
encoder_test_rows: list[dict[str, Any]] = []
for metric_index, metric in enumerate(run_level_metrics):
    pivot = (
        statistical_encoder_runs.pivot_table(index=["k", "seed"], columns="model", values=metric, aggfunc="first")
        .reindex(columns=["BERTopic", "BERTopic-ParsBERT"])
        .dropna()
    )
    if len(pivot) != expected_statistical_blocks:
        raise RuntimeError(f"Incomplete BERTopic encoder blocks for {metric}: {len(pivot)}")
    differences = pivot["BERTopic-ParsBERT"] - pivot["BERTopic"]
    try:
        statistic, raw_p = wilcoxon(
            pivot["BERTopic-ParsBERT"],
            pivot["BERTopic"],
            zero_method="wilcox",
            alternative="two-sided",
            method="auto",
        )
    except ValueError:
        statistic, raw_p = 0.0, 1.0
    difference_frame = pivot.reset_index()[["k", "seed"]].copy()
    difference_frame["difference"] = differences.to_numpy(dtype=float)
    ci_low, ci_high = cluster_bootstrap_mean_ci(
        difference_frame,
        value_column="difference",
        cluster_column="k",
        seed=RANDOM_SEED + 120_000 + metric_index,
    )
    encoder_test_rows.append({
        "analysis": "bertopic_encoder_sensitivity",
        "metric": metric,
        "model_a": "BERTopic-ParsBERT",
        "model_b": "BERTopic",
        "paired_blocks": len(differences),
        "mean_difference_a_minus_b": float(differences.mean()),
        "cluster_bootstrap_95_ci_low": ci_low,
        "cluster_bootstrap_95_ci_high": ci_high,
        "wilcoxon_statistic": float(statistic),
        "raw_p": float(raw_p),
        "rank_biserial": rank_biserial_from_differences(differences),
        "alpha": ALPHA,
        "inference_scope": "Encoder sensitivity conditional on fixed BERTopic/UMAP/KMeans/c-TF-IDF settings.",
    })
encoder_adjusted = holm_adjust([row["raw_p"] for row in encoder_test_rows])
for row, adjusted_p in zip(encoder_test_rows, encoder_adjusted):
    row["holm_p"] = adjusted_p
    row["significant_holm_0_05"] = bool(adjusted_p < ALPHA)
bertopic_encoder_tests = pd.DataFrame(encoder_test_rows)

print("\nFriedman tests")
display(friedman_results)
print("\nWilcoxon pairwise tests with Holm correction, rank-biserial effects, and 5,000-resample CIs")
display(pairwise_topic_results)
print("\nBERTopic encoder sensitivity tests")
display(bertopic_encoder_tests)

# =============================================================================
# 7. TRUE PREPROCESSING ABLATION FROM RAW TEXT
# =============================================================================

heading("4. Preprocessing ablation")
ablation_rows: list[dict[str, Any]] = []
if RUN_PREPROCESSING_ABLATION:
    ablation_train_source = train.sample(n=ABLATION_TRAIN_DOCS, random_state=RANDOM_SEED).reset_index(drop=True)
    ablation_test_source = topic_validation.sample(n=ABLATION_TEST_DOCS, random_state=RANDOM_SEED).reset_index(drop=True)
    variants = {
        "full": dict(normalize_text=True, remove_stopwords=True, lemmatize=True),
        "no_lemmatization": dict(normalize_text=True, remove_stopwords=True, lemmatize=False),
        "no_stopword_removal": dict(normalize_text=True, remove_stopwords=False, lemmatize=True),
        "normalization_tokenization_only": dict(normalize_text=True, remove_stopwords=False, lemmatize=False),
    }
    for variant_name, options in variants.items():
        train_outputs = [
            preprocess_text_with_reason(text, **options)
            for text in tqdm(ablation_train_source["raw_text"], desc=f"Ablation {variant_name} train")
        ]
        test_outputs = [
            preprocess_text_with_reason(text, **options)
            for text in tqdm(ablation_test_source["raw_text"], desc=f"Ablation {variant_name} test")
        ]
        variant_train = [text for text, _reason in train_outputs if text]
        variant_test = [text for text, _reason in test_outputs if text]
        if len(variant_train) < int(0.95 * ABLATION_TRAIN_DOCS) or len(variant_test) < int(0.95 * ABLATION_TEST_DOCS):
            raise RuntimeError(
                f"Ablation {variant_name} retained only {len(variant_train)} train and {len(variant_test)} test documents."
            )
        token_train = [text.split() for text in variant_train]
        token_test = [text.split() for text in variant_test]
        dictionary = Dictionary(token_train)
        dictionary.filter_extremes(no_below=3, no_above=0.8)
        if len(dictionary) < ABLATION_K * TOP_N_WORDS:
            raise RuntimeError(f"Ablation {variant_name} dictionary is too small: {len(dictionary)}")
        corpus = [dictionary.doc2bow(tokens) for tokens in token_train]
        variant_reference = Dictionary(token_test)
        for seed in SEEDS:
            set_seed(seed)
            started = time.time()
            model = LdaModel(
                corpus=corpus,
                id2word=dictionary,
                num_topics=ABLATION_K,
                random_state=seed,
                passes=LDA_PASSES,
                iterations=LDA_ITERATIONS,
                alpha=LDA_ALPHA,
                eta=LDA_ETA,
                minimum_probability=0.0,
                eval_every=None,
            )
            raw_topics = [
                [word for word, _ in model.show_topic(topic_id, topn=TOPIC_CANDIDATE_WORDS)]
                for topic_id in range(ABLATION_K)
            ]
            metrics, _canonical_topics = evaluate_topics_against_reference(
                raw_topics,
                token_test,
                variant_reference,
            )
            if not all(math.isfinite(metrics[column]) for column in ("c_v", "c_npmi", "diversity", "distinctness")):
                raise RuntimeError(f"Non-finite ablation metrics for {variant_name}, seed={seed}: {metrics}")
            ablation_rows.append({
                "variant": variant_name,
                "seed": seed,
                "k": ABLATION_K,
                "train_documents_requested": ABLATION_TRAIN_DOCS,
                "validation_documents_requested": ABLATION_TEST_DOCS,
                "train_documents_retained": len(variant_train),
                "validation_documents_retained": len(variant_test),
                "c_v": metrics["c_v"],
                "c_npmi": metrics["c_npmi"],
                "diversity": metrics["diversity"],
                "distinctness": metrics["distinctness"],
                "valid_topic_count": metrics["valid_topic_count"],
                "training_seconds": time.time() - started,
            })
            del model

ablation_runs = pd.DataFrame(ablation_rows)
ablation_summary_rows: list[dict[str, Any]] = []
for variant, group in ablation_runs.groupby("variant", sort=True):
    row: dict[str, Any] = {
        "variant": variant,
        "runs": len(group),
        "k": ABLATION_K,
        "train_documents_retained": int(group["train_documents_retained"].iloc[0]),
        "validation_documents_retained": int(group["validation_documents_retained"].iloc[0]),
        "valid_topic_count_mean": group["valid_topic_count"].mean(),
    }
    for metric_index, metric in enumerate(("c_v", "c_npmi", "diversity", "distinctness")):
        ci_low, ci_high = bootstrap_mean_ci(group[metric], seed=RANDOM_SEED + 200_000 + metric_index + len(ablation_summary_rows))
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_sd"] = group[metric].std(ddof=1)
        row[f"{metric}_bootstrap_ci_low"] = ci_low
        row[f"{metric}_bootstrap_ci_high"] = ci_high
    ablation_summary_rows.append(row)
ablation_summary = pd.DataFrame(ablation_summary_rows)

ablation_friedman_rows: list[dict[str, Any]] = []
ablation_pairwise_rows: list[dict[str, Any]] = []
ablation_variant_order = [
    "full", "no_lemmatization", "no_stopword_removal", "normalization_tokenization_only"
]
for metric_index, metric in enumerate(("c_v", "c_npmi", "diversity", "distinctness")):
    pivot = ablation_runs.pivot(index="seed", columns="variant", values=metric).reindex(columns=ablation_variant_order).dropna()
    if len(pivot) != len(SEEDS):
        raise RuntimeError(f"Incomplete ablation blocks for {metric}: {len(pivot)}")
    statistic, p_value = friedmanchisquare(*[pivot[variant].to_numpy(dtype=float) for variant in ablation_variant_order])
    if not math.isfinite(float(statistic)) or not math.isfinite(float(p_value)):
        statistic, p_value = 0.0, 1.0
    ablation_friedman_rows.append({
        "metric": metric, "block_unit": "seed", "blocks": len(pivot),
        "variants": ", ".join(ablation_variant_order),
        "friedman_chi_square": float(statistic), "raw_p": float(p_value),
        "kendalls_w": float(statistic / (len(pivot) * (len(ablation_variant_order) - 1))) if len(pivot) else None,
        "alpha": ALPHA, "significant_0_05": bool(p_value < ALPHA),
        "inference_scope": "Exploratory ablation inference with only three seeds; low statistical power.",
    })
    temporary: list[dict[str, Any]] = []
    for pair_index, (left, right) in enumerate(combinations(ablation_variant_order, 2)):
        differences = pivot[left] - pivot[right]
        try:
            w_stat, raw_p = wilcoxon(pivot[left], pivot[right], zero_method="wilcox", alternative="two-sided", method="auto")
        except ValueError:
            w_stat, raw_p = 0.0, 1.0
        ci_low, ci_high = bootstrap_mean_ci(
            differences, seed=RANDOM_SEED + 230_000 + 1000 * metric_index + pair_index
        )
        temporary.append({
            "metric": metric, "variant_a": left, "variant_b": right,
            "paired_seeds": len(differences),
            "mean_difference_a_minus_b": float(differences.mean()),
            "bootstrap_95_ci_low": ci_low, "bootstrap_95_ci_high": ci_high,
            "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
            "wilcoxon_statistic": float(w_stat), "raw_p": float(raw_p),
            "rank_biserial": rank_biserial_from_differences(differences),
            "alpha": ALPHA,
            "inference_scope": "Exploratory; three paired seeds only.",
        })
    adjusted = holm_adjust([row["raw_p"] for row in temporary])
    for row, adjusted_p in zip(temporary, adjusted):
        row["holm_p"] = adjusted_p
        row["significant_holm_0_05"] = bool(adjusted_p < ALPHA)
        ablation_pairwise_rows.append(row)
ablation_friedman_results = pd.DataFrame(ablation_friedman_rows)
ablation_pairwise_tests = pd.DataFrame(ablation_pairwise_rows)

print("\nPreprocessing-ablation run-level results")
display(ablation_runs)
print("\nPreprocessing-ablation aggregate results")
display(ablation_summary)
print("\nPreprocessing-ablation Friedman tests")
display(ablation_friedman_results)
print("\nPreprocessing-ablation pairwise tests with Holm correction")
display(ablation_pairwise_tests)

# =============================================================================
# 8. STANZA NER — NO DUMMY FALLBACKS
# =============================================================================

heading("5. Persian named-entity recognition")
ALLOWED_ENTITY_TYPES = ("Person", "Location", "Organization", "Product", "Facility", "Event")
ENTITY_TYPE_MAP = {
    "LOC": "Location", "LOCATION": "Location", "GPE": "Location",
    "PER": "Person", "PERS": "Person", "PERSON": "Person",
    "ORG": "Organization", "ORGANIZATION": "Organization",
    "FAC": "Facility", "FACILITY": "Facility",
    "PRODUCT": "Product", "PRO": "Product",
    "EVENT": "Event",
}


def normalize_entity_surface(text: str) -> str:
    normalized = normalize_base_text(text, use_hazm=True).replace("\u200c", " ")
    return WHITESPACE_RE.sub(" ", normalized).strip()


stanza_dir = TEMP_DIR / "stanza_resources"
stanza.download("fa", processors="tokenize,ner", model_dir=str(stanza_dir), verbose=False)
ner_pipeline = stanza.Pipeline(
    lang="fa",
    processors="tokenize,ner",
    model_dir=str(stanza_dir),
    use_gpu=STANZA_USE_GPU and torch.cuda.is_available(),
    ner_batch_size=STANZA_NER_BATCH_SIZE,
    verbose=False,
)


def run_ner(
    frame: pd.DataFrame,
    split_name: str,
    max_docs: int,
) -> tuple[list[list[tuple[str, str]]], pd.DataFrame, dict[str, Any], pd.DataFrame]:
    subset = frame.iloc[:max_docs].reset_index(drop=True)
    if len(subset) != max_docs:
        raise RuntimeError(f"NER expected {max_docs} {split_name} documents, obtained {len(subset)}.")

    doc_entities: list[list[tuple[str, str]]] = []
    mention_rows: list[dict[str, Any]] = []
    errors: list[dict[str, Any]] = []
    excluded_type_counter: Counter[str] = Counter()
    documents_with_entities = 0

    for position, row in tqdm(subset.iterrows(), total=len(subset), desc=f"Stanza NER {split_name}"):
        raw_text = str(row["raw_text"])[:NER_MAX_CHARS_PER_DOC]
        try:
            document = ner_pipeline(raw_text)
            mentions: list[tuple[str, str]] = []
            for entity in document.ents:
                raw_type = str(entity.type).upper().replace("B-", "").replace("I-", "")
                entity_type = ENTITY_TYPE_MAP.get(raw_type)
                if entity_type not in ALLOWED_ENTITY_TYPES:
                    excluded_type_counter[raw_type] += 1
                    continue
                surface = normalize_entity_surface(entity.text)
                if not surface:
                    continue
                typed = (surface, entity_type)
                mentions.append(typed)
                mention_rows.append({
                    "split": split_name,
                    "doc_position": int(position),
                    "doc_id": int(row["doc_id"]),
                    "surface": surface,
                    "entity_type": entity_type,
                    "raw_ner_type": str(entity.type),
                })
            if mentions:
                documents_with_entities += 1
            doc_entities.append(mentions)
        except Exception as exc:
            doc_entities.append([])
            errors.append({
                "split": split_name,
                "doc_position": int(position),
                "doc_id": int(row["doc_id"]),
                "error": repr(exc),
            })

    mentions = pd.DataFrame(mention_rows)
    errors_frame = pd.DataFrame(errors)
    error_rate = len(errors) / len(subset)
    if error_rate >= MAX_ACCEPTABLE_NER_ERROR_RATE:
        raise RuntimeError(
            f"NER error rate for {split_name} was {error_rate:.4%}; protocol requires < {MAX_ACCEPTABLE_NER_ERROR_RATE:.2%}."
        )
    if mentions.empty:
        raise RuntimeError(f"Stanza returned zero accepted entities for {split_name}; no fallback data are allowed.")
    if not set(mentions["entity_type"]).issubset(ALLOWED_ENTITY_TYPES):
        raise RuntimeError(f"Unexpected entity types remained in {split_name}: {set(mentions['entity_type'])}")

    diagnostics = {
        "split": split_name,
        "documents_processed": len(subset),
        "documents_with_entities": documents_with_entities,
        "documents_with_errors": len(errors),
        "error_rate": error_rate,
        "max_characters_per_document": NER_MAX_CHARS_PER_DOC,
        "total_mentions": len(mentions),
        "unique_surface_forms": int(mentions["surface"].nunique()),
        "unique_typed_surface_nodes": int(mentions[["surface", "entity_type"]].drop_duplicates().shape[0]),
        "excluded_unknown_type_mentions": int(sum(excluded_type_counter.values())),
        "excluded_unknown_type_counts": dict(excluded_type_counter),
        "dummy_or_fallback_entities_used": False,
    }
    return doc_entities, mentions, diagnostics, errors_frame


train_doc_entities, train_mentions, train_ner_diagnostics, train_ner_errors = run_ner(
    train, "train", NER_TRAIN_DOCS
)
test_doc_entities, test_mentions, test_ner_diagnostics, test_ner_errors = run_ner(
    test, "link_test", NER_TEST_DOCS
)

ner_type_summary = pd.DataFrame({"entity_type": ALLOWED_ENTITY_TYPES})
train_type_counts = train_mentions["entity_type"].value_counts()
test_type_counts = test_mentions["entity_type"].value_counts()
ner_type_summary["train_mentions"] = ner_type_summary["entity_type"].map(train_type_counts).fillna(0).astype(int)
ner_type_summary["test_mentions"] = ner_type_summary["entity_type"].map(test_type_counts).fillna(0).astype(int)
ner_type_summary["train_percent"] = 100 * ner_type_summary["train_mentions"] / ner_type_summary["train_mentions"].sum()
ner_type_summary["test_percent"] = 100 * ner_type_summary["test_mentions"] / ner_type_summary["test_mentions"].sum()

train_surface_counts = (
    train_mentions.groupby(["surface", "entity_type"], as_index=False)
    .size()
    .rename(columns={"size": "mentions"})
    .sort_values(["mentions", "surface"], ascending=[False, True])
)
test_surface_counts = (
    test_mentions.groupby(["surface", "entity_type"], as_index=False)
    .size()
    .rename(columns={"size": "mentions"})
    .sort_values(["mentions", "surface"], ascending=[False, True])
)
top_entity_surfaces = train_surface_counts.head(100).reset_index(drop=True)
top_entity_surfaces_test = test_surface_counts.head(100).reset_index(drop=True)
top_entity_surfaces_by_type = (
    train_surface_counts.sort_values(["entity_type", "mentions", "surface"], ascending=[True, False, True])
    .groupby("entity_type", as_index=False, group_keys=False)
    .head(20)
    .reset_index(drop=True)
)
top_entity_surfaces_by_type_test = (
    test_surface_counts.sort_values(["entity_type", "mentions", "surface"], ascending=[True, False, True])
    .groupby("entity_type", as_index=False, group_keys=False)
    .head(20)
    .reset_index(drop=True)
)
ner_summary = pd.DataFrame([train_ner_diagnostics, test_ner_diagnostics])
ner_errors = pd.concat([train_ner_errors, test_ner_errors], ignore_index=True)

# Deterministic, unscored audit template. It supports later human validation
# without fabricating precision/recall or agreement statistics.
def make_ner_audit_template(target_rows: int = 200) -> pd.DataFrame:
    combined = pd.concat([train_mentions, test_mentions], ignore_index=True)
    source_lookup = pd.concat([
        train[["doc_id", "raw_text"]].assign(split="train"),
        test[["doc_id", "raw_text"]].assign(split="link_test"),
    ], ignore_index=True)
    samples: list[pd.DataFrame] = []
    groups = list(combined.groupby(["split", "entity_type"], sort=True))
    per_group = max(1, math.ceil(target_rows / max(len(groups), 1)))
    for group_index, ((_split, _type), group) in enumerate(groups):
        samples.append(group.sample(n=min(per_group, len(group)), random_state=RANDOM_SEED + group_index))
    audit = pd.concat(samples, ignore_index=True).head(target_rows)
    audit = audit.merge(source_lookup, on=["split", "doc_id"], how="left", validate="many_to_one")
    audit["document_excerpt"] = audit["raw_text"].astype(str).str.slice(0, 700)
    audit = audit.drop(columns="raw_text")
    audit["boundary_correct"] = ""
    audit["type_correct"] = ""
    audit["canonical_entity_or_alias"] = ""
    audit["annotator"] = ""
    audit["notes"] = ""
    return audit.sort_values(["split", "entity_type", "doc_id", "surface"]).reset_index(drop=True)


ner_audit_template = make_ner_audit_template()
stanza_resource_hashes = sha256_directory(stanza_dir)

ner_document_rows: list[dict[str, Any]] = []
for split_name, source_frame, entities_by_doc, error_frame in (
    ("train", train.iloc[:NER_TRAIN_DOCS].reset_index(drop=True), train_doc_entities, train_ner_errors),
    ("link_test", test.iloc[:NER_TEST_DOCS].reset_index(drop=True), test_doc_entities, test_ner_errors),
):
    error_positions = set(error_frame["doc_position"].astype(int)) if not error_frame.empty else set()
    for position, entities in enumerate(entities_by_doc):
        ner_document_rows.append({
            "split": split_name,
            "doc_position": position,
            "doc_id": int(source_frame.iloc[position]["doc_id"]),
            "mention_count": len(entities),
            "unique_typed_entity_count": len(set(entities)),
            "has_entity": bool(entities),
            "processing_error": position in error_positions,
        })
ner_document_summary = pd.DataFrame(ner_document_rows)
entity_surface_frequency_all = pd.concat([
    train_surface_counts.assign(split="train"),
    test_surface_counts.assign(split="link_test"),
], ignore_index=True)[["split", "surface", "entity_type", "mentions"]].sort_values(
    ["split", "mentions", "surface"], ascending=[True, False, True]
).reset_index(drop=True)

print("\nNER diagnostics")
display(ner_summary)
print("\nEntity-type distribution")
display(ner_type_summary)
print("\nTop entity surface forms overall")
display(top_entity_surfaces.head(30))
print("\nTop training entity surface forms by type")
display(top_entity_surfaces_by_type)
print("\nTop test entity surface forms overall")
display(top_entity_surfaces_test.head(30))
print("\nTop test entity surface forms by type")
display(top_entity_surfaces_by_type_test)
print(f"\nComplete NER document diagnostics contain {len(ner_document_summary):,} rows and are saved in the final JSON evidence.")
display(ner_document_summary.head(200))
print("\nUnscored NER human-audit template (complete template saved in JSON)")
display(ner_audit_template.head(30))
if not ner_errors.empty:
    print("\nNER document errors (error rate remained below 1%)")
    display(ner_errors)

# =============================================================================
# 9. TRAIN-ONLY ENTITY GRAPH AND TOPIC VECTORS
# =============================================================================

heading("6. Entity-association graph")

mention_frequency: Counter[tuple[str, str]] = Counter()
document_frequency: Counter[tuple[str, str]] = Counter()
for mentions in train_doc_entities:
    mention_frequency.update(mentions)
    document_frequency.update(set(mentions))
ranked_entities = [entity for entity, _count in document_frequency.most_common(MAX_GRAPH_NODES)]
entity_to_id = {entity: idx for idx, entity in enumerate(ranked_entities)}

edge_counts: Counter[tuple[int, int]] = Counter()
doc_entity_ids: list[list[int]] = []
for mentions in train_doc_entities:
    unique_ids = {entity_to_id[entity] for entity in set(mentions) if entity in entity_to_id}
    ordered_ids = sorted(unique_ids, key=lambda entity_id: (-document_frequency[ranked_entities[entity_id]], entity_id))
    ordered_ids = ordered_ids[:MAX_ENTITIES_PER_DOCUMENT]
    doc_entity_ids.append(ordered_ids)
    for left, right in combinations(sorted(ordered_ids), 2):
        edge_counts[(left, right)] += 1

train_graph = nx.Graph()
for entity_id, (surface, entity_type) in enumerate(ranked_entities):
    train_graph.add_node(
        entity_id,
        surface=surface,
        entity_type=entity_type,
        mention_frequency=int(mention_frequency[(surface, entity_type)]),
        document_frequency=int(document_frequency[(surface, entity_type)]),
    )
for (left, right), weight in edge_counts.items():
    train_graph.add_edge(left, right, weight=int(weight))

entity_inventory = pd.DataFrame([
    {
        "entity_id": entity_id,
        "surface": surface,
        "entity_type": entity_type,
        "mention_frequency": int(mention_frequency[(surface, entity_type)]),
        "document_frequency": int(document_frequency[(surface, entity_type)]),
    }
    for entity_id, (surface, entity_type) in enumerate(ranked_entities)
])

if train_graph.number_of_nodes() < LINK_PREDICTION_NODES:
    raise RuntimeError("The training graph has fewer nodes than the requested evaluation graph.")


def dense_document_topic_matrix(model: LdaModel, token_lists: Sequence[Sequence[str]]) -> np.ndarray:
    matrix = np.zeros((len(token_lists), model.num_topics), dtype=np.float32)
    for index, tokens in enumerate(tqdm(token_lists, desc=f"LDA K={model.num_topics} document-topic inference")):
        bow = model.id2word.doc2bow(tokens)
        distribution = model.get_document_topics(bow, minimum_probability=0.0)
        for topic_id, probability in distribution:
            matrix[index, topic_id] = probability
    return matrix


def entity_topic_vectors_for_model(model: LdaModel) -> np.ndarray:
    doc_topics = dense_document_topic_matrix(model, train_tokens)
    sums = np.zeros((len(ranked_entities), model.num_topics), dtype=np.float64)
    counts = np.zeros(len(ranked_entities), dtype=np.int64)
    for doc_index, entity_ids in enumerate(doc_entity_ids):
        for entity_id in entity_ids:
            sums[entity_id] += doc_topics[doc_index]
            counts[entity_id] += 1
    vectors = np.zeros_like(sums, dtype=np.float32)
    nonzero = counts > 0
    vectors[nonzero] = (sums[nonzero] / counts[nonzero, None]).astype(np.float32)
    return normalize(vectors, norm="l2", copy=False).astype(np.float32)


entity_topic_vectors_by_k: dict[int, np.ndarray] = {}
for k in GRAPH_TOPIC_K_SENSITIVITY:
    if k not in selected_lda_models:
        raise RuntimeError(f"Required graph LDA model K={k} was not retained.")
    entity_topic_vectors_by_k[k] = entity_topic_vectors_for_model(selected_lda_models[k])

components = list(nx.connected_components(train_graph))
component_sizes = np.asarray(sorted((len(component) for component in components), reverse=True), dtype=float)
degrees = np.asarray([degree for _node, degree in train_graph.degree()], dtype=float)
edge_weights = np.asarray([data.get("weight", 1.0) for _u, _v, data in train_graph.edges(data=True)], dtype=float)
if len(degrees) == 0 or len(edge_weights) == 0:
    raise RuntimeError("The train-only entity graph is empty.")

graph_summary = pd.DataFrame([{
    "source_split": "training only",
    "all_typed_nodes_before_cap": len(document_frequency),
    "retained_graph_nodes": train_graph.number_of_nodes(),
    "node_cap": MAX_GRAPH_NODES,
    "train_edges": train_graph.number_of_edges(),
    "density": nx.density(train_graph),
    "connected_components": len(components),
    "giant_component_nodes": int(component_sizes.max()),
    "giant_component_fraction": float(component_sizes.max() / train_graph.number_of_nodes()),
    "isolates": nx.number_of_isolates(train_graph),
    "mean_degree": float(degrees.mean()),
    "degree_sd": float(degrees.std(ddof=1)),
    "median_degree": float(np.median(degrees)),
    "max_degree": int(degrees.max()),
    "mean_edge_weight": float(edge_weights.mean()),
    "edge_weight_sd": float(edge_weights.std(ddof=1)),
    "median_edge_weight": float(np.median(edge_weights)),
    "max_edge_weight": int(edge_weights.max()),
    "approximate_average_clustering": nx.approximation.average_clustering(
        train_graph,
        trials=min(2_000, max(100, train_graph.number_of_nodes())),
        seed=RANDOM_SEED,
    ),
}])

def distribution_summary(values: np.ndarray, variable: str) -> pd.DataFrame:
    quantiles = [0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.0]
    return pd.DataFrame({
        "variable": variable,
        "quantile": quantiles,
        "value": [float(np.quantile(values, quantile)) for quantile in quantiles],
    })

graph_degree_distribution = distribution_summary(degrees, "node_degree")
graph_edge_weight_distribution = distribution_summary(edge_weights, "edge_weight")
graph_component_distribution = distribution_summary(component_sizes, "component_size")

graph_type_summary = pd.DataFrame({"entity_type": ALLOWED_ENTITY_TYPES})
graph_type_counts = Counter(train_graph.nodes[node]["entity_type"] for node in train_graph.nodes)
graph_type_summary["retained_nodes"] = graph_type_summary["entity_type"].map(graph_type_counts).fillna(0).astype(int)
graph_type_summary["percent"] = 100 * graph_type_summary["retained_nodes"] / graph_type_summary["retained_nodes"].sum()

graph_topic_vector_summary = pd.DataFrame([
    {
        "topic_k": k,
        "entities": vectors.shape[0],
        "vector_dimensions": vectors.shape[1],
        "finite_values": bool(np.isfinite(vectors).all()),
        "nonzero_entity_vectors": int((np.linalg.norm(vectors, axis=1) > 0).sum()),
        "construction": "mean training-document LDA topic distribution per typed entity, then L2 normalization",
    }
    for k, vectors in sorted(entity_topic_vectors_by_k.items())
])
if not graph_topic_vector_summary["finite_values"].all():
    raise RuntimeError("Non-finite entity-topic vectors detected.")

print("\nGraph summary")
display(graph_summary)
print(f"\nEntity inventory contains {len(entity_inventory):,} typed surface-form nodes; complete table is saved in the final JSON evidence.")
display(entity_inventory.head(200))
print("\nGraph node-type distribution")
display(graph_type_summary)
print("\nDegree distribution")
display(graph_degree_distribution)
print("\nEdge-weight distribution")
display(graph_edge_weight_distribution)
print("\nConnected-component size distribution")
display(graph_component_distribution)
print("\nEntity-topic vector sensitivity summary")
display(graph_topic_vector_summary)

# =============================================================================
# 10. LEAKAGE-CONTROLLED REPEATED NOVEL-LINK PREDICTION
# =============================================================================

heading("7. Leakage-controlled novel-link prediction")


def canonical_pair(left: int, right: int) -> tuple[int, int]:
    if left == right:
        raise ValueError("Self-pairs are not valid graph links.")
    return (left, right) if left < right else (right, left)


def hash_pair_set(pairs: Sequence[tuple[int, int]]) -> str:
    digest = hashlib.sha256()
    for left, right in sorted(pairs):
        digest.update(f"{int(left)},{int(right)}\n".encode("utf-8"))
    return digest.hexdigest()


def pairs_from_documents(doc_entities: Sequence[Sequence[tuple[str, str]]], allowed_nodes: set[int]) -> set[tuple[int, int]]:
    pairs: set[tuple[int, int]] = set()
    for mentions in doc_entities:
        node_ids = sorted({entity_to_id[entity] for entity in set(mentions) if entity in entity_to_id and entity_to_id[entity] in allowed_nodes})
        node_ids = node_ids[:MAX_ENTITIES_PER_DOCUMENT]
        for left, right in combinations(node_ids, 2):
            pairs.add(canonical_pair(left, right))
    return pairs


def weighted_node2vec_walk(graph: nx.Graph, start: int, walk_length: int, p: float, q: float, rng: np.random.Generator) -> list[str]:
    walk = [start]
    while len(walk) < walk_length:
        current = walk[-1]
        neighbors = list(graph.neighbors(current))
        if not neighbors:
            break
        if len(walk) == 1:
            weights = np.asarray([graph[current][neighbor].get("weight", 1.0) for neighbor in neighbors], dtype=float)
        else:
            previous = walk[-2]
            weights = []
            for neighbor in neighbors:
                edge_weight = float(graph[current][neighbor].get("weight", 1.0))
                if neighbor == previous:
                    bias = 1.0 / p
                elif graph.has_edge(previous, neighbor):
                    bias = 1.0
                else:
                    bias = 1.0 / q
                weights.append(edge_weight * bias)
            weights = np.asarray(weights, dtype=float)
        probabilities = weights / weights.sum() if weights.sum() > 0 else np.full(len(neighbors), 1 / len(neighbors))
        walk.append(int(rng.choice(neighbors, p=probabilities)))
    return [str(node) for node in walk]


def train_walk_embeddings(graph: nx.Graph, p: float, q: float, seed: int) -> dict[int, np.ndarray]:
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    walks: list[list[str]] = []
    for _ in range(WALKS_PER_NODE):
        rng.shuffle(nodes)
        for node in nodes:
            walks.append(weighted_node2vec_walk(graph, int(node), WALK_LENGTH, p, q, rng))
    model = Word2Vec(
        sentences=walks,
        vector_size=EMBEDDING_DIM,
        window=WORD2VEC_WINDOW,
        min_count=1,
        sg=1,
        workers=1,
        epochs=WORD2VEC_EPOCHS,
        seed=seed,
    )
    return {int(node): model.wv[str(node)] for node in graph.nodes if str(node) in model.wv}


def cosine_score(left: np.ndarray, right: np.ndarray) -> float:
    denominator = float(np.linalg.norm(left) * np.linalg.norm(right))
    return float(np.dot(left, right) / denominator) if denominator else 0.0


def build_degree_bins(graph: nx.Graph, nodes: Sequence[int], n_bins: int) -> tuple[dict[int, int], dict[int, list[int]]]:
    degrees_series = pd.Series({int(node): graph.degree(node) for node in nodes}, dtype=float)
    try:
        labels = pd.qcut(degrees_series.rank(method="first"), q=min(n_bins, len(nodes)), labels=False, duplicates="drop")
    except ValueError:
        labels = pd.Series(0, index=degrees_series.index)
    node_to_bin = {int(node): int(label) for node, label in labels.items()}
    bin_to_nodes: dict[int, list[int]] = defaultdict(list)
    for node, label in node_to_bin.items():
        bin_to_nodes[label].append(node)
    return node_to_bin, dict(bin_to_nodes)


def degree_bin_matched_negatives(
    positives: Sequence[tuple[int, int]],
    forbidden: set[tuple[int, int]],
    graph: nx.Graph,
    nodes: Sequence[int],
    count: int,
    seed: int,
) -> list[tuple[int, int]]:
    rng = np.random.default_rng(seed)
    node_to_bin, bin_to_nodes = build_degree_bins(graph, nodes, DEGREE_MATCH_BINS)
    negatives: set[tuple[int, int]] = set()
    attempts = 0
    max_attempts = max(200_000, count * 500)
    positive_cycle = itertools.cycle(positives)
    while len(negatives) < count and attempts < max_attempts:
        source_left, source_right = next(positive_cycle)
        left_pool = bin_to_nodes.get(node_to_bin[source_left], list(nodes))
        right_pool = bin_to_nodes.get(node_to_bin[source_right], list(nodes))
        left = int(rng.choice(left_pool))
        right = int(rng.choice(right_pool))
        attempts += 1
        if left == right:
            continue
        pair = canonical_pair(left, right)
        if pair not in forbidden and pair not in negatives:
            negatives.add(pair)
    if len(negatives) < count:
        raise RuntimeError(f"Could sample only {len(negatives)} of {count} degree-matched negatives.")
    return sorted(negatives)


def adamic_adar_score(graph: nx.Graph, left: int, right: int) -> float:
    common = set(graph.neighbors(left)) & set(graph.neighbors(right))
    score = 0.0
    for node in common:
        degree = graph.degree(node)
        if degree > 1:
            score += 1.0 / math.log(degree)
    return score


ranked_evaluation_nodes = sorted(
    train_graph.nodes,
    key=lambda node: (-train_graph.nodes[node].get("document_frequency", 0), node),
)[:LINK_PREDICTION_NODES]
allowed_nodes = set(int(node) for node in ranked_evaluation_nodes)
evaluation_graph = train_graph.subgraph(ranked_evaluation_nodes).copy()
train_edges = {canonical_pair(int(left), int(right)) for left, right in evaluation_graph.edges}
all_test_pairs = pairs_from_documents(test_doc_entities, allowed_nodes)
all_forbidden = train_edges | all_test_pairs

link_rows: list[dict[str, Any]] = []
link_fold_assignment_rows: list[dict[str, Any]] = []
link_evaluation_diagnostic_rows: list[dict[str, Any]] = []
for repeat in range(LINK_REPEATS):
    embedding_seed = RANDOM_SEED + 1_000 * repeat
    deepwalk_embeddings = train_walk_embeddings(evaluation_graph, p=1.0, q=1.0, seed=embedding_seed)
    node2vec_embeddings = train_walk_embeddings(evaluation_graph, p=NODE2VEC_P, q=NODE2VEC_Q, seed=embedding_seed)

    fold_splits = list(
        KFold(n_splits=LINK_FOLDS, shuffle=True, random_state=embedding_seed)
        .split(np.arange(len(test_doc_entities)))
    )
    doc_to_fold: dict[int, int] = {}
    for fold, (_unused, doc_indices) in enumerate(fold_splits, start=1):
        for document_position in doc_indices:
            doc_to_fold[int(document_position)] = fold
            link_fold_assignment_rows.append({
                "repeat": repeat + 1,
                "fold": fold,
                "evaluation_id": f"R{repeat + 1}F{fold}",
                "link_test_doc_position": int(document_position),
                "doc_id": int(test.iloc[int(document_position)]["doc_id"]),
            })

    # Assign each novel test pair to exactly one fold in which it occurs. This
    # prevents the same positive edge from appearing in multiple folds of a repeat.
    candidate_folds_by_pair: dict[tuple[int, int], set[int]] = defaultdict(set)
    for document_position, mentions in enumerate(test_doc_entities):
        fold = doc_to_fold[document_position]
        for pair in pairs_from_documents([mentions], allowed_nodes) - train_edges:
            candidate_folds_by_pair[pair].add(fold)

    positive_pairs_by_fold: dict[int, list[tuple[int, int]]] = {fold: [] for fold in range(1, LINK_FOLDS + 1)}
    for pair, candidate_folds in candidate_folds_by_pair.items():
        candidates = sorted(candidate_folds)
        digest = hashlib.sha256(f"{repeat}|{pair[0]}|{pair[1]}".encode("utf-8")).digest()
        assigned_fold = candidates[int.from_bytes(digest[:8], "big") % len(candidates)]
        positive_pairs_by_fold[assigned_fold].append(pair)

    used_positive_pairs: set[tuple[int, int]] = set()
    used_negative_pairs: set[tuple[int, int]] = set()
    for fold, (_unused, doc_indices) in enumerate(fold_splits, start=1):
        positives = sorted(positive_pairs_by_fold[fold])
        rng = np.random.default_rng(embedding_seed + fold)
        if len(positives) > MAX_POSITIVE_EDGES_PER_FOLD:
            chosen = rng.choice(len(positives), size=MAX_POSITIVE_EDGES_PER_FOLD, replace=False)
            positives = [positives[index] for index in chosen]
        if len(positives) < 20:
            raise RuntimeError(f"Repeat {repeat + 1}, fold {fold} has only {len(positives)} unique novel positives.")
        positives = sorted(set(positives))
        if set(positives) & used_positive_pairs:
            raise RuntimeError("Positive test pairs were reused across folds within a repeat.")

        negatives = degree_bin_matched_negatives(
            positives=positives,
            forbidden=all_forbidden | used_negative_pairs,
            graph=evaluation_graph,
            nodes=ranked_evaluation_nodes,
            count=len(positives) * NEGATIVE_TO_POSITIVE_RATIO,
            seed=embedding_seed + 10_000 + fold,
        )
        positive_set = set(positives)
        negative_set = set(negatives)
        positive_train_overlap = len(positive_set & train_edges)
        negative_forbidden_overlap = len(negative_set & all_forbidden)
        positive_negative_overlap = len(positive_set & negative_set)
        positive_reuse_within_repeat = len(positive_set & used_positive_pairs)
        negative_reuse_within_repeat = len(negative_set & used_negative_pairs)
        self_pair_count = sum(left == right for left, right in positive_set | negative_set)
        if (
            positive_train_overlap or negative_forbidden_overlap or positive_negative_overlap
            or positive_reuse_within_repeat or negative_reuse_within_repeat or self_pair_count
        ):
            raise RuntimeError(
                "Link-prediction leakage/integrity failure: "
                f"positive_train_overlap={positive_train_overlap}, "
                f"negative_forbidden_overlap={negative_forbidden_overlap}, "
                f"positive_negative_overlap={positive_negative_overlap}, "
                f"positive_reuse={positive_reuse_within_repeat}, negative_reuse={negative_reuse_within_repeat}, "
                f"self_pairs={self_pair_count}"
            )
        used_positive_pairs.update(positive_set)
        used_negative_pairs.update(negative_set)
        link_evaluation_diagnostic_rows.append({
            "repeat": repeat + 1,
            "fold": fold,
            "evaluation_id": f"R{repeat + 1}F{fold}",
            "fold_documents": len(doc_indices),
            "positive_edges": len(positives),
            "negative_edges": len(negatives),
            "positive_pair_sha256": hash_pair_set(positives),
            "negative_pair_sha256": hash_pair_set(negatives),
            "positive_train_overlap": positive_train_overlap,
            "negative_forbidden_overlap": negative_forbidden_overlap,
            "positive_negative_overlap": positive_negative_overlap,
            "positive_reuse_within_repeat": positive_reuse_within_repeat,
            "negative_reuse_within_repeat": negative_reuse_within_repeat,
            "self_pair_count": self_pair_count,
        })
        pairs = positives + negatives
        labels = np.asarray([1] * len(positives) + [0] * len(negatives), dtype=int)
        score_sets: dict[str, list[float]] = {
            f"Topic-K{k}": [] for k in GRAPH_TOPIC_K_SENSITIVITY
        }
        score_sets.update({
            "Node2Vec": [], "DeepWalk": [], "Jaccard": [],
            "Adamic-Adar": [], "Preferential-Attachment": [], "Random": [],
        })
        random_scores = np.random.default_rng(embedding_seed + 20_000 + fold).random(len(pairs))
        for pair_index, (left, right) in enumerate(pairs):
            for k, vectors in entity_topic_vectors_by_k.items():
                score_sets[f"Topic-K{k}"].append(float(np.dot(vectors[left], vectors[right])))
            score_sets["Node2Vec"].append(cosine_score(node2vec_embeddings[left], node2vec_embeddings[right]))
            score_sets["DeepWalk"].append(cosine_score(deepwalk_embeddings[left], deepwalk_embeddings[right]))
            left_neighbors = set(evaluation_graph.neighbors(left))
            right_neighbors = set(evaluation_graph.neighbors(right))
            union = left_neighbors | right_neighbors
            score_sets["Jaccard"].append(len(left_neighbors & right_neighbors) / len(union) if union else 0.0)
            score_sets["Adamic-Adar"].append(adamic_adar_score(evaluation_graph, left, right))
            score_sets["Preferential-Attachment"].append(float(evaluation_graph.degree(left) * evaluation_graph.degree(right)))
            score_sets["Random"].append(float(random_scores[pair_index]))
        for method, scores in score_sets.items():
            link_rows.append({
                "repeat": repeat + 1,
                "fold": fold,
                "evaluation_id": f"R{repeat + 1}F{fold}",
                "method": method,
                "roc_auc": float(roc_auc_score(labels, scores)),
                "average_precision": float(average_precision_score(labels, scores)),
                "positive_edges": len(positives),
                "negative_edges": len(negatives),
                "evaluation_nodes": evaluation_graph.number_of_nodes(),
                "training_edges": evaluation_graph.number_of_edges(),
                "positive_train_overlap": positive_train_overlap,
                "negative_forbidden_overlap": negative_forbidden_overlap,
                "positive_negative_overlap": positive_negative_overlap,
                "positive_reuse_within_repeat": positive_reuse_within_repeat,
                "negative_reuse_within_repeat": negative_reuse_within_repeat,
                "self_pair_count": self_pair_count,
            })

link_folds = pd.DataFrame(link_rows)
link_fold_assignments = pd.DataFrame(link_fold_assignment_rows)
link_evaluation_diagnostics = pd.DataFrame(link_evaluation_diagnostic_rows)
expected_link_methods = [
    "Topic-K100", "Topic-K200", "Topic-K300",
    "Node2Vec", "DeepWalk", "Jaccard", "Adamic-Adar",
    "Preferential-Attachment", "Random",
]
expected_link_rows = LINK_REPEATS * LINK_FOLDS * len(expected_link_methods)
if len(link_folds) != expected_link_rows:
    raise RuntimeError(f"Expected {expected_link_rows} link rows, obtained {len(link_folds)}.")
if set(link_folds["method"]) != set(expected_link_methods):
    raise RuntimeError(f"Unexpected link methods: {sorted(link_folds['method'].unique())}")
if not np.isfinite(link_folds[["roc_auc", "average_precision"]].to_numpy(dtype=float)).all():
    raise RuntimeError("Non-finite link-prediction metrics detected.")
if link_folds[["positive_train_overlap", "negative_forbidden_overlap", "positive_negative_overlap", "positive_reuse_within_repeat", "negative_reuse_within_repeat", "self_pair_count"]].to_numpy().sum() != 0:
    raise RuntimeError("Link-prediction leakage diagnostics are non-zero.")

link_repeat_means = (
    link_folds.groupby(["repeat", "method"], as_index=False)[["roc_auc", "average_precision"]]
    .mean()
    .sort_values(["repeat", "method"])
    .reset_index(drop=True)
)

link_summary_rows: list[dict[str, Any]] = []
for method_index, method in enumerate(expected_link_methods):
    fold_group = link_folds[link_folds["method"] == method].copy()
    repeat_group = link_repeat_means[link_repeat_means["method"] == method].copy()
    if len(fold_group) != LINK_REPEATS * LINK_FOLDS or len(repeat_group) != LINK_REPEATS:
        raise RuntimeError(
            f"Method {method} has {len(fold_group)} fold evaluations and {len(repeat_group)} repeat means; "
            f"expected {LINK_REPEATS * LINK_FOLDS} and {LINK_REPEATS}."
        )
    auc_low, auc_high = bootstrap_mean_ci(
        repeat_group["roc_auc"], seed=RANDOM_SEED + 300_000 + method_index
    )
    ap_low, ap_high = bootstrap_mean_ci(
        repeat_group["average_precision"], seed=RANDOM_SEED + 310_000 + method_index
    )
    link_summary_rows.append({
        "method": method,
        "fold_evaluations": len(fold_group),
        "independent_repeat_blocks": len(repeat_group),
        "repeats": LINK_REPEATS,
        "folds_per_repeat": LINK_FOLDS,
        "roc_auc_mean": repeat_group["roc_auc"].mean(),
        "roc_auc_repeat_sd": repeat_group["roc_auc"].std(ddof=1),
        "roc_auc_fold_sd_descriptive": fold_group["roc_auc"].std(ddof=1),
        "roc_auc_repeat_bootstrap_ci_low": auc_low,
        "roc_auc_repeat_bootstrap_ci_high": auc_high,
        "average_precision_mean": repeat_group["average_precision"].mean(),
        "average_precision_repeat_sd": repeat_group["average_precision"].std(ddof=1),
        "average_precision_fold_sd_descriptive": fold_group["average_precision"].std(ddof=1),
        "average_precision_repeat_bootstrap_ci_low": ap_low,
        "average_precision_repeat_bootstrap_ci_high": ap_high,
        "bootstrap_unit": "repeat mean",
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
    })
link_summary = pd.DataFrame(link_summary_rows).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)

# Inferential tests use five repeat-level means. Folds within a repeat are
# cross-validation components, not independent inferential observations.
link_friedman_rows: list[dict[str, Any]] = []
link_pairwise_rows: list[dict[str, Any]] = []
for metric_index, metric in enumerate(("roc_auc", "average_precision")):
    pivot = (
        link_repeat_means.pivot(index="repeat", columns="method", values=metric)
        .reindex(columns=expected_link_methods)
        .dropna()
    )
    if len(pivot) != LINK_REPEATS:
        raise RuntimeError(f"Incomplete repeat-level link blocks for {metric}: {len(pivot)}")
    statistic, p_value = friedmanchisquare(*[pivot[method].to_numpy(dtype=float) for method in expected_link_methods])
    link_friedman_rows.append({
        "metric": metric,
        "block_unit": "repeat mean across five document folds",
        "blocks": len(pivot),
        "methods": ", ".join(expected_link_methods),
        "friedman_chi_square": float(statistic),
        "raw_p": float(p_value),
        "kendalls_w": float(statistic / (len(pivot) * (len(expected_link_methods) - 1))),
        "alpha": ALPHA,
        "significant_0_05": bool(p_value < ALPHA),
        "inference_scope": "Exploratory repeat-level comparison; only five repeat blocks are available.",
    })

    temporary: list[dict[str, Any]] = []
    pivot_reset = pivot.reset_index()
    for pair_index, (left, right) in enumerate(combinations(expected_link_methods, 2)):
        differences = pivot_reset[left] - pivot_reset[right]
        try:
            statistic_pair, raw_p = wilcoxon(
                pivot_reset[left],
                pivot_reset[right],
                zero_method="wilcox",
                alternative="two-sided",
                method="auto",
            )
        except ValueError:
            statistic_pair, raw_p = 0.0, 1.0
        ci_low, ci_high = bootstrap_mean_ci(
            differences, seed=RANDOM_SEED + 320_000 + 1000 * metric_index + pair_index
        )
        non_ties = differences[differences != 0]
        sign_p = (
            binomtest(int((non_ties > 0).sum()), n=len(non_ties), p=0.5, alternative="two-sided").pvalue
            if len(non_ties) else 1.0
        )
        temporary.append({
            "metric": metric,
            "method_a": left,
            "method_b": right,
            "paired_repeats": len(differences),
            "mean_difference_a_minus_b": float(differences.mean()),
            "repeat_bootstrap_95_ci_low": ci_low,
            "repeat_bootstrap_95_ci_high": ci_high,
            "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
            "a_wins": int((differences > 0).sum()),
            "ties": int((differences == 0).sum()),
            "a_losses": int((differences < 0).sum()),
            "wilcoxon_statistic": float(statistic_pair),
            "wilcoxon_raw_p": float(raw_p),
            "sign_test_raw_p": float(sign_p),
            "rank_biserial": rank_biserial_from_differences(differences),
            "alpha": ALPHA,
            "inference_scope": "Exploratory; inference uses five repeat means, while fold-level rows remain descriptive.",
        })
    wilcoxon_adjusted = holm_adjust([row["wilcoxon_raw_p"] for row in temporary])
    sign_adjusted = holm_adjust([row["sign_test_raw_p"] for row in temporary])
    for row, adjusted_w, adjusted_sign in zip(temporary, wilcoxon_adjusted, sign_adjusted):
        row["wilcoxon_holm_p"] = adjusted_w
        row["sign_test_holm_p"] = adjusted_sign
        row["significant_wilcoxon_holm_0_05"] = bool(adjusted_w < ALPHA)
        row["significant_sign_test_holm_0_05"] = bool(adjusted_sign < ALPHA)
        link_pairwise_rows.append(row)

link_friedman_results = pd.DataFrame(link_friedman_rows)
link_pairwise_tests = pd.DataFrame(link_pairwise_rows)
main_method = f"Topic-K{GRAPH_TOPIC_K_MAIN}"
main_vs_baselines = link_pairwise_tests[
    (link_pairwise_tests["method_a"] == main_method) | (link_pairwise_tests["method_b"] == main_method)
].copy()

print("\nAll repeated-fold link-prediction results")
display(link_folds)
print("\nLink-prediction evaluation diagnostics and pair-set hashes")
display(link_evaluation_diagnostics)
print(f"\nExact fold assignment evidence contains {len(link_fold_assignments):,} rows and is saved in the final JSON evidence.")
display(link_fold_assignments.head(200))
print("\nLink-prediction method summaries based on repeat means")
display(link_summary)
print("\nLink-prediction Friedman tests")
display(link_friedman_results)
print("\nAll pairwise Wilcoxon tests with Holm correction")
display(link_pairwise_tests)
print(f"\nPredeclared {main_method} comparisons")
display(main_vs_baselines)

# =============================================================================
# 11. CLAIM-READY SUMMARY, INTEGRITY CHECKS, AND TWO FINAL FILES
# =============================================================================


heading("8. Claim-ready summary and integrity checks")


def extract_pairwise_result(
    frame: pd.DataFrame,
    metric: str,
    item_a: str,
    item_b: str,
    left_column: str,
    right_column: str,
    difference_column: str,
    low_column: str,
    high_column: str,
    p_column: str,
) -> dict[str, Any] | None:
    subset = frame[frame["metric"] == metric]
    mask = (
        ((subset[left_column] == item_a) & (subset[right_column] == item_b))
        | ((subset[left_column] == item_b) & (subset[right_column] == item_a))
    )
    if not mask.any():
        return None
    row = subset.loc[mask].iloc[0]
    orientation = 1.0 if row[left_column] == item_a else -1.0
    low = float(row[low_column])
    high = float(row[high_column])
    if orientation < 0:
        low, high = -high, -low
    return {
        "item_a": item_a,
        "item_b": item_b,
        "mean_difference_a_minus_b": orientation * float(row[difference_column]),
        "ci_low": low,
        "ci_high": high,
        "adjusted_p": float(row[p_column]),
        "significant": bool(float(row[p_column]) < ALPHA),
        "rank_biserial_a_minus_b": orientation * float(row["rank_biserial"]),
    }


best_by_model_display = best_configs[[
    "model", "model_family", "benchmark_role", "encoder", "k",
    "c_v_mean", "c_v_sd", "c_v_bootstrap_ci_low", "c_v_bootstrap_ci_high",
    "c_npmi_mean", "c_npmi_sd", "c_npmi_bootstrap_ci_low", "c_npmi_bootstrap_ci_high",
    "diversity_mean", "diversity_sd", "distinctness_mean", "distinctness_sd",
    "stability", "valid_topic_fraction_mean", "reference_match_rate_mean",
    "training_seconds_mean",
]].copy()

common_k_display = common_k_results.copy()
highest_core = core_best_configs.sort_values("c_v_mean", ascending=False).iloc[0]
second_core = core_best_configs.sort_values("c_v_mean", ascending=False).iloc[1]
common_k_winner = common_k_results.sort_values("c_v_mean", ascending=False).iloc[0]

# Inferential tests concern average paired performance across the fixed
# statistical grid; they are never used to validate model-specific maxima.
grid_mean_cv = (
    statistical_core_runs.groupby("model", as_index=False)["c_v"].mean()
    .sort_values("c_v", ascending=False)
    .reset_index(drop=True)
)
grid_first, grid_second = grid_mean_cv.iloc[0], grid_mean_cv.iloc[1]
grid_cv_pair = extract_pairwise_result(
    pairwise_topic_results,
    metric="c_v",
    item_a=str(grid_first["model"]),
    item_b=str(grid_second["model"]),
    left_column="model_a",
    right_column="model_b",
    difference_column="mean_difference_a_minus_b",
    low_column="cluster_bootstrap_95_ci_low",
    high_column="cluster_bootstrap_95_ci_high",
    p_column="holm_p",
)

main_link_row = link_summary[link_summary["method"] == main_method].iloc[0]
strongest_baseline = (
    link_summary[~link_summary["method"].str.startswith("Topic-K")]
    .sort_values("roc_auc_mean", ascending=False)
    .iloc[0]
)
link_auc_pair = extract_pairwise_result(
    link_pairwise_tests,
    metric="roc_auc",
    item_a=main_method,
    item_b=str(strongest_baseline["method"]),
    left_column="method_a",
    right_column="method_b",
    difference_column="mean_difference_a_minus_b",
    low_column="repeat_bootstrap_95_ci_low",
    high_column="repeat_bootstrap_95_ci_high",
    p_column="wilcoxon_holm_p",
)

encoder_cv_row = bertopic_encoder_tests[bertopic_encoder_tests["metric"] == "c_v"].iloc[0]

claim_rows: list[dict[str, Any]] = [
    {
        "area": "dataset",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"The complete source file contained {raw_row_count:,} rows. After language eligibility screening and "
            f"text-hash deduplication, {len(unique_candidate_meta):,} unique Persian-rich documents were eligible."
        ),
        "statistically_validated": None,
        "scope": "Descriptive source-corpus audit.",
    },
    {
        "area": "sampling",
        "statement_type": "methodological fact",
        "claim_ready_statement": (
            f"A proportional raw-length-stratified random sample of {len(sample):,} documents was drawn from the full "
            "eligible corpus; the sample was not selected as the longest documents."
        ),
        "statistically_validated": None,
        "scope": "Reproducible sampling procedure conditional on seed 42.",
    },
    {
        "area": "topic modeling",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"At the predeclared common topic count K={PRIMARY_COMMON_K}, the four model families were compared under "
            "the same held-out corpus and top-word protocol; this matched-K table is the primary model comparison."
        ),
        "statistically_validated": False,
        "scope": "Primary matched-K descriptive comparison; model-specific optima are secondary sensitivity analyses.",
    },
    {
        "area": "topic modeling",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"At K={PRIMARY_COMMON_K}, {common_k_winner['model']} had the highest mean C_v "
            f"({common_k_winner['c_v_mean']:.4f}); NPMI, diversity, distinctness, and stability are reported beside "
            "coherence so this matched-K result is not interpreted as universal dominance."
        ),
        "statistically_validated": False,
        "scope": "Primary common-K descriptive result with explicit multi-metric qualification.",
    },
    {
        "area": "topic modeling",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"When each core model was allowed to choose its own coherence-maximizing K, "
            f"{highest_core['model']} achieved the highest mean C_v ({highest_core['c_v_mean']:.4f}) "
            f"at K={int(highest_core['k'])}. This is a metric-specific optimum, not an overall winner; "
            f"diversity, distinctness, NPMI, and stability must be interpreted jointly."
        ),
        "statistically_validated": False,
        "scope": "Descriptive metric-specific selection; no composite quality score is used.",
    },
    {
        "area": "topic modeling",
        "statement_type": "exploratory inference",
        "claim_ready_statement": (
            f"Across the {expected_statistical_blocks} matched K × seed configurations in the fixed statistical grid, "
            f"{grid_first['model']} had the highest mean C_v ({grid_first['c_v']:.4f}); its mean difference from "
            f"{grid_second['model']} was {grid_cv_pair['mean_difference_a_minus_b']:.4f} "
            f"(K-cluster bootstrap 95% CI {grid_cv_pair['ci_low']:.4f} to {grid_cv_pair['ci_high']:.4f}; "
            f"Holm-adjusted Wilcoxon p={grid_cv_pair['adjusted_p']:.4g})."
            if grid_cv_pair else "The planned paired C_v comparison was unavailable."
        ),
        "statistically_validated": grid_cv_pair["significant"] if grid_cv_pair else False,
        "scope": "Exploratory paired inference across the fixed statistical K grid; model-specific maxima are descriptive only.",
    },
    {
        "area": "BERTopic encoder sensitivity",
        "statement_type": "exploratory inference",
        "claim_ready_statement": (
            f"Under fixed BERTopic settings, mean-pooled ParsBERT minus multilingual MiniLM produced a mean C_v "
            f"difference of {encoder_cv_row['mean_difference_a_minus_b']:.4f} "
            f"(K-cluster bootstrap 95% CI {encoder_cv_row['cluster_bootstrap_95_ci_low']:.4f} to "
            f"{encoder_cv_row['cluster_bootstrap_95_ci_high']:.4f}; Holm-adjusted p={encoder_cv_row['holm_p']:.4g})."
        ),
        "statistically_validated": bool(encoder_cv_row["significant_holm_0_05"]),
        "scope": "Conditional on mean pooling, truncation length, UMAP, KMeans, and c-TF-IDF parameters.",
    },
    {
        "area": "NER",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"Stanza extracted {len(train_mentions):,} accepted training mentions and {len(test_mentions):,} accepted "
            f"link-test mentions from {NER_TRAIN_DOCS:,} training and {NER_TEST_DOCS:,} link-test documents; document-level error rates were "
            f"{train_ner_diagnostics['error_rate']:.3%} and {test_ner_diagnostics['error_rate']:.3%}, respectively."
        ),
        "statistically_validated": None,
        "scope": "Typed normalized surface forms; no entity linking or canonical disambiguation.",
    },
    {
        "area": "entity graph",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"The training-only graph retained {train_graph.number_of_nodes():,} typed surface-form nodes and "
            f"{train_graph.number_of_edges():,} document co-occurrence edges."
        ),
        "statistically_validated": None,
        "scope": "Entity association graph, not a multi-relational knowledge graph.",
    },
    {
        "area": "link prediction",
        "statement_type": "observation",
        "claim_ready_statement": (
            f"The predeclared {main_method} method achieved mean ROC-AUC {main_link_row['roc_auc_mean']:.4f} "
            f"and mean average precision {main_link_row['average_precision_mean']:.4f}; means and uncertainty were "
            f"computed from {LINK_REPEATS} repeat-level blocks, each averaging {LINK_FOLDS} document folds."
        ),
        "statistically_validated": False,
        "scope": "Descriptive repeated cross-validation result.",
    },
    {
        "area": "link prediction",
        "statement_type": "exploratory inference",
        "claim_ready_statement": (
            f"Relative to the strongest non-topic baseline ({strongest_baseline['method']}), {main_method} had a mean "
            f"ROC-AUC difference of {link_auc_pair['mean_difference_a_minus_b']:.4f} "
            f"(repeat-level bootstrap 95% CI {link_auc_pair['ci_low']:.4f} to {link_auc_pair['ci_high']:.4f}; "
            f"Holm-adjusted Wilcoxon p={link_auc_pair['adjusted_p']:.4g})."
            if link_auc_pair else "The planned link-prediction comparison was unavailable."
        ),
        "statistically_validated": link_auc_pair["significant"] if link_auc_pair else False,
        "scope": "Exploratory repeat-level inference based on five repeat means; all evidence derives from Persian Wikipedia.",
    },
]
claim_ready_summary = pd.DataFrame(claim_rows)

limitations = pd.DataFrame([
    {
        "limitation": "Hash-based deduplication",
        "statement": "Corpus-scale deduplication uses pandas' stable 64-bit text hash for tractability. Collisions are extremely unlikely but are not cryptographically impossible; selected-document provenance retains the hash for audit.",
    },
    {
        "limitation": "Shared-source evaluation",
        "statement": "Preprocessing, topics, named entities, graph edges, and held-out links originate from the same Wikipedia source, although topic validation and link testing use mutually exclusive documents; domain alignment may still increase compatibility.",
    },
    {
        "limitation": "Domain specificity",
        "statement": "Results describe Persian Wikipedia and do not establish population-level generalization to social media, news, conversational text, or specialized domains.",
    },
    {
        "limitation": "CTM training budget",
        "statement": f"CTM uses the official CombinedTM.fit loop for ten epochs and the shared {len(shared_vocabulary):,}-term vocabulary; conclusions remain conditional on this computational budget.",
    },
    {
        "limitation": "ParsBERT pooling",
        "statement": "ParsBERT is a masked-language-model checkpoint converted to document embeddings through mean pooling; it is not a Persian sentence-similarity model fine-tuned for semantic clustering.",
    },
    {
        "limitation": "Encoder/parameter conditionality",
        "statement": "BERTopic conclusions are conditional on the selected encoders, 256-token truncation, UMAP, KMeans, Persian whitespace tokenization, and the official ClassTfidfTransformer configuration.",
    },
    {
        "limitation": "No human validation",
        "statement": "This pipeline contains no human topic-interpretability or edge-validity annotations and makes no inter-rater-reliability claim.",
    },
    {
        "limitation": "Three-seed uncertainty",
        "statement": "Topic-model means and seed-bootstrap intervals are based on only three predeclared random seeds; they characterize initialization sensitivity but are not population confidence intervals.",
    },
    {
        "limitation": "Repeated-fold inference",
        "statement": "Repeated cross-validation folds overlap; Wilcoxon tests and bootstrap intervals are exploratory rather than independent population-level inference.",
    },
    {
        "limitation": "NER truncation",
        "statement": f"NER processes at most the first {NER_MAX_CHARS_PER_DOC:,} characters of each document; entities appearing later in long documents are not observed.",
    },
    {
        "limitation": "Head-entity graph",
        "statement": f"The graph retains the {MAX_GRAPH_NODES:,} most frequent typed nodes, at most {MAX_ENTITIES_PER_DOCUMENT} retained entities per document, and the link evaluation uses the {LINK_PREDICTION_NODES:,} highest-document-frequency nodes; conclusions concern the graph head rather than the full entity tail.",
    },
    {
        "limitation": "Near-duplicate documents",
        "statement": "Exact text-hash duplicates are removed, but semantic or near-duplicate pages and revisions are not explicitly clustered; residual similarity may remain across train and test.",
    },
    {
        "limitation": "Surface-form entities",
        "statement": "Nodes are typed normalized surface forms. Homonyms are not disambiguated and aliases are not linked to canonical entities.",
    },
])

# Integrity checks that must pass before evidence files are finalized.
top_longest_rows = set(
    unique_candidate_meta.nlargest(WORKING_SAMPLE_SIZE, "raw_chars")["source_row"].astype(int)
)
selected_rows = set(sampled_meta["source_row"].astype(int))
population_stratum_proportions = unique_candidate_meta["raw_length_stratum"].value_counts(normalize=True).sort_index()
sample_stratum_proportions = sampled_meta["raw_length_stratum"].value_counts(normalize=True).sort_index()
max_stratum_proportion_difference = float(
    (population_stratum_proportions - sample_stratum_proportions).abs().max()
)
pre_output_checks = {
    "full_dataset_scanned_raw_row_count_positive": raw_row_count > 0,
    "full_dataset_second_pass_completed": row_offset == raw_row_count,
    "sampling_is_random_stratified_not_top_n_longest": selected_rows != top_longest_rows,
    "sampling_covers_all_populated_raw_length_strata": set(sampled_meta["raw_length_stratum"]) == set(unique_candidate_meta["raw_length_stratum"]),
    "sampling_preserves_stratum_proportions_within_0_5_percent": max_stratum_proportion_difference <= 0.005,
    "working_sample_size_50000": len(sample) == WORKING_SAMPLE_SIZE,
    "train_size_expected": len(train) == TRAIN_SIZE,
    "topic_validation_size_expected": len(topic_validation) == TOPIC_VALIDATION_SIZE,
    "link_test_size_expected": len(test) == LINK_TEST_SIZE,
    "all_analysis_role_overlaps_zero": not any(split_overlaps.values()),
    "core_four_models_present": set(core_runs["model"]) == {"LDA", "NMF", "CTM", "BERTopic"},
    "core_topic_runs_complete": len(core_runs) == expected_core_runs,
    "parsbert_sensitivity_runs_complete": len(all_runs[all_runs["model"] == "BERTopic-ParsBERT"]) == expected_parsbert_extra_runs,
    "total_topic_runs_complete": len(all_runs) == expected_total_runs,
    "all_core_metrics_finite": bool(np.isfinite(all_runs[core_metric_columns].to_numpy(dtype=float)).all()),
    "bertopic_two_encoders_present": set(bertopic_encoder_runs["encoder"].dropna()) == {"multilingual-MiniLM", "ParsBERT-mean-pooling"},
    "bertopic_all_runs_use_ctfidf": bool(bertopic_encoder_runs["c_tfidf_used"].fillna(False).all()),
    "bertopic_all_coherence_values_valid": bool(
        np.isfinite(bertopic_encoder_runs[["c_v", "c_npmi"]].to_numpy(dtype=float)).all()
        and (bertopic_encoder_runs["valid_topic_fraction"] >= 0.95).all()
    ),
    "ctm_uses_configured_training_documents": bool((core_runs.loc[core_runs["model"] == "CTM", "training_documents"] == CTM_TRAIN_DOCS).all()),
    "ctm_uses_configured_epochs": bool((core_runs.loc[core_runs["model"] == "CTM", "epochs"] == CTM_EPOCHS).all()),
    "ctm_uses_official_fit_loop": bool((core_runs.loc[core_runs["model"] == "CTM", "training_implementation"] == "official CombinedTM.fit").all()),
    "shared_vocabulary_size_within_cap": 100 <= len(shared_vocabulary) <= SHARED_VOCAB_SIZE,
    "all_core_models_record_shared_vocabulary": bool(
        all_runs.loc[all_runs["benchmark_role"] == "core", "shared_vocabulary_size"].eq(len(shared_vocabulary)).all()
    ),
    "ctm_vocabulary_matches_shared_vocabulary": bool((core_runs.loc[core_runs["model"] == "CTM", "controlled_bow_vocabulary"] == len(shared_vocabulary)).all()),
    "topic_friedman_tests_present": len(friedman_results) >= 5,
    "topic_pairwise_tests_holm_complete": bool(len(pairwise_topic_results) > 0 and pairwise_topic_results["holm_p"].notna().all()),
    "bootstrap_resamples_5000": BOOTSTRAP_RESAMPLES == 5_000,
    "statistical_grid_is_subset_of_descriptive_grid": set(STATISTICAL_TOPIC_COUNTS).issubset(TOPIC_COUNTS),
    "statistical_blocks_dynamic_and_complete": len(statistical_core_runs) == 4 * expected_statistical_blocks,
    "model_revisions_are_commit_like": all(bool(re.fullmatch(r"[0-9a-fA-F]{7,64}", revision)) for revision in MODEL_REVISIONS.values()),
    "stanza_resource_hashes_present": bool(stanza_resource_hashes),
    "package_versions_match_reference_environment": not PACKAGE_VERSION_MISMATCHES,
    "preprocessing_ablation_4x3_complete": len(ablation_runs) == 4 * len(SEEDS),
    "preprocessing_ablation_metrics_finite": bool(np.isfinite(ablation_runs[["c_v", "c_npmi", "diversity", "distinctness"]].to_numpy(dtype=float)).all()),
    "preprocessing_ablation_statistics_complete": len(ablation_friedman_results) == 4 and len(ablation_pairwise_tests) == 4 * math.comb(4, 2),
    "ner_train_documents_configured": train_ner_diagnostics["documents_processed"] == NER_TRAIN_DOCS,
    "ner_link_test_documents_configured": test_ner_diagnostics["documents_processed"] == NER_TEST_DOCS,
    "ner_train_error_rate_below_1_percent": train_ner_diagnostics["error_rate"] < 0.01,
    "ner_test_error_rate_below_1_percent": test_ner_diagnostics["error_rate"] < 0.01,
    "ner_no_fabricated_or_fallback_entities": not train_ner_diagnostics["dummy_or_fallback_entities_used"] and not test_ner_diagnostics["dummy_or_fallback_entities_used"],
    "ner_only_six_predeclared_types": set(train_mentions["entity_type"]) == set(ALLOWED_ENTITY_TYPES) and set(test_mentions["entity_type"]) == set(ALLOWED_ENTITY_TYPES),
    "graph_constructed_from_training_entities_only": True,
    "graph_node_cap_respected": train_graph.number_of_nodes() <= MAX_GRAPH_NODES,
    "graph_has_1000_evaluation_nodes": evaluation_graph.number_of_nodes() == LINK_PREDICTION_NODES,
    "graph_topic_sensitivity_k100_200_300_complete": set(entity_topic_vectors_by_k) == set(GRAPH_TOPIC_K_SENSITIVITY),
    "link_prediction_expected_evaluations_per_method": bool((link_folds.groupby("method").size() == LINK_REPEATS * LINK_FOLDS).all()),
    "link_prediction_repeat_means_complete": bool((link_repeat_means.groupby("method").size() == LINK_REPEATS).all()),
    "link_fold_assignments_complete": len(link_fold_assignments) == LINK_TEST_SIZE * LINK_REPEATS,
    "link_evaluation_diagnostics_complete": len(link_evaluation_diagnostics) == LINK_REPEATS * LINK_FOLDS,
    "link_pair_hashes_present": bool(
        link_evaluation_diagnostics[["positive_pair_sha256", "negative_pair_sha256"]].apply(
            lambda column: column.astype(str).str.fullmatch(r"[0-9a-f]{64}").all()
        ).all()
    ),
    "link_prediction_all_9_methods_present": set(link_folds["method"]) == set(expected_link_methods),
    "link_prediction_metrics_finite": bool(np.isfinite(link_folds[["roc_auc", "average_precision"]].to_numpy(dtype=float)).all()),
    "link_prediction_no_positive_train_overlap": int(link_folds["positive_train_overlap"].sum()) == 0,
    "link_prediction_no_forbidden_negatives": int(link_folds["negative_forbidden_overlap"].sum()) == 0,
    "link_prediction_no_self_class_or_fold_reuse": int(link_folds[["positive_negative_overlap", "positive_reuse_within_repeat", "negative_reuse_within_repeat", "self_pair_count"]].to_numpy().sum()) == 0,
    "link_friedman_tests_present": len(link_friedman_results) == 2,
    "link_pairwise_holm_tests_complete": bool(len(link_pairwise_tests) == 2 * math.comb(len(expected_link_methods), 2) and link_pairwise_tests["wilcoxon_holm_p"].notna().all()),
}
failed_pre_output = [name for name, passed in pre_output_checks.items() if not passed]
if failed_pre_output:
    raise RuntimeError(f"Pre-output integrity checks failed: {failed_pre_output}")

integrity_checks = dict(pre_output_checks)
integrity_checks.update({
    "json_file_created_and_valid": False,
    "markdown_report_created_and_valid": False,
    "output_directory_contains_only_two_public_files": False,
})

# Dataset hashing is intentionally performed once and recorded in the protocol.
log("Computing SHA-256 for the source dataset...")
dataset_sha256 = sha256_file(DATASET_FILE)


def build_protocol() -> dict[str, Any]:
    return {
        "run_note": RUN_NOTE,
        "run_started_utc": datetime.fromtimestamp(START_TIME, tz=timezone.utc).isoformat(),
        "run_finished_utc": datetime.now(timezone.utc).isoformat(),
        "elapsed_hours": (time.time() - START_TIME) / 3600,
        "alpha": ALPHA,
        "bootstrap_resamples": BOOTSTRAP_RESAMPLES,
        "dataset": {
            "public_source": "Kaggle dataset amirpourmand/fa-wikipedia",
            "local_file_name": DATASET_FILE.name,
            "file_size_bytes": DATASET_FILE.stat().st_size,
            "sha256": dataset_sha256,
            "text_column": TEXT_COLUMN,
            "title_column": TITLE_COLUMN,
            "raw_rows_scanned": raw_row_count,
        },
        "sampling": {
            "method": "full-corpus scan, stable uint64 text-hash deduplication, proportional raw-length-stratified random sampling",
            "deduplication_hash": "pandas.util.hash_pandas_object uint64",
            "not_top_n_longest": True,
            "raw_length_quantile_edges": raw_strata_edges.tolist(),
            "clean_token_length_quantile_edges": clean_strata_edges.tolist(),
            "working_sample_size": WORKING_SAMPLE_SIZE,
            "train_size": TRAIN_SIZE,
            "topic_validation_size": TOPIC_VALIDATION_SIZE,
            "link_test_size": LINK_TEST_SIZE,
            "random_seed": RANDOM_SEED,
            "role_separation": "topic_validation is used for coherence/NPMI and CTM inference; link_test is reserved for NER/link prediction",
            "maximum_absolute_stratum_proportion_difference": max_stratum_proportion_difference,
            "sample_provenance_rows_recorded": len(sample_provenance),
        },
        "topic_benchmark": {
            "core_models": ["LDA", "NMF", "CTM", "BERTopic"],
            "core_runs": expected_core_runs,
            "bertopic_encoder_sensitivity_extra_runs": expected_parsbert_extra_runs,
            "total_runs": expected_total_runs,
            "topic_counts_descriptive": TOPIC_COUNTS,
            "topic_counts_statistical": STATISTICAL_TOPIC_COUNTS,
            "seeds": SEEDS,
            "held_out_reference_documents": len(reference_texts),
            "top_n_words": TOP_N_WORDS,
            "shared_training_vocabulary": {
                "size": len(shared_vocabulary),
                "min_df": SHARED_VOCAB_MIN_DF,
                "max_df": SHARED_VOCAB_MAX_DF,
                "used_by": ["LDA", "NMF", "CTM", "BERTopic"],
            },
            "candidate_words_before_reference_matching": TOPIC_CANDIDATE_WORDS,
            "metrics": ["C_v", "NPMI", "diversity", "distinctness", "stability", "runtime", "reference coverage"],
            "composite_quality_score_used": False,
            "lda": {
                "passes": LDA_PASSES,
                "iterations": LDA_ITERATIONS,
                "alpha": LDA_ALPHA,
                "eta": LDA_ETA,
            },
            "nmf": {
                "init": NMF_INIT,
                "solver": "mu",
                "max_iter": NMF_MAX_ITER,
                "tol": NMF_TOL,
            },
            "ctm": {
                "architecture": "CombinedTM",
                "training_documents": CTM_TRAIN_DOCS,
                "test_documents": CTM_TEST_DOCS,
                "epochs": CTM_EPOCHS,
                "batch_size": CTM_BATCH_SIZE,
                "controlled_bow_max_features": CTM_MAX_FEATURES,
                "contextual_encoder": MULTILINGUAL_ENCODER_MODEL,
                "contextual_embeddings_l2_normalized": False,
                "training_implementation": "official CombinedTM.fit loop",
            },
            "bertopic": {
                "core_encoder": MULTILINGUAL_ENCODER_MODEL,
                "sensitivity_encoder": PARSBERT_ENCODER_MODEL,
                "parsbert_pooling": "mean pooling added through SentenceTransformers",
                "max_sequence_length": 256,
                "clusterer": "KMeans",
                "kmeans_n_init": BERTOPIC_KMEANS_N_INIT,
                "kmeans_max_iter": BERTOPIC_KMEANS_MAX_ITER,
                "manual_topic_interface": "BaseEmbedder + BaseDimensionalityReduction + BaseCluster; labels passed through y",
                "ctfidf": "official BERTopic ClassTfidfTransformer(reduce_frequent_words=True)",
                "lexical_input": "clean Persian unigrams",
                "contextual_input": "normalized raw Persian text",
                "contextual_embeddings_l2_normalized": False,
            },
            "statistics": {
                "friedman": True,
                "pairwise_wilcoxon": True,
                "holm_bonferroni": True,
                "rank_biserial_effect_size": True,
                "bootstrap_ci": "5,000 resamples; K-cluster bootstrap for topic-model differences",
                "alpha": ALPHA,
            },
        },
        "preprocessing_ablation": {
            "training_documents": ABLATION_TRAIN_DOCS,
            "test_documents": ABLATION_TEST_DOCS,
            "k": ABLATION_K,
            "variants": ["full", "no_lemmatization", "no_stopword_removal", "normalization_tokenization_only"],
        },
        "ner": {
            "tool": "Stanza Persian tokenize,ner pipeline",
            "train_documents": NER_TRAIN_DOCS,
            "test_documents": NER_TEST_DOCS,
            "max_characters_per_document": NER_MAX_CHARS_PER_DOC,
            "accepted_types": ALLOWED_ENTITY_TYPES,
            "maximum_error_rate_exclusive": MAX_ACCEPTABLE_NER_ERROR_RATE,
            "dummy_fallback_allowed": False,
        },
        "graph": {
            "node_definition": "typed normalized surface form (surface, entity_type)",
            "source": "training entities only",
            "max_graph_nodes": MAX_GRAPH_NODES,
            "max_entities_per_document": MAX_ENTITIES_PER_DOCUMENT,
            "edge_weight": "training-document co-occurrence count",
            "topic_vector_k_sensitivity": GRAPH_TOPIC_K_SENSITIVITY,
            "topic_vector_aggregation": "mean training-document LDA distribution per entity, L2 normalized",
        },
        "link_prediction": {
            "evaluation_nodes": LINK_PREDICTION_NODES,
            "evaluation_node_selection": "highest training-document-frequency nodes within the retained graph",
            "positive_pairs_disjoint_within_repeat": True,
            "negative_pairs_disjoint_within_repeat": True,
            "folds": LINK_FOLDS,
            "repeats": LINK_REPEATS,
            "evaluations_per_method": LINK_FOLDS * LINK_REPEATS,
            "methods": expected_link_methods,
            "novel_positive_definition": "link-test-document co-occurrence absent from the training graph",
            "negative_definition": "degree-bin-matched pair absent from the training graph and every link-test positive set",
            "metrics": ["ROC-AUC", "average precision"],
            "node2vec": {"p": NODE2VEC_P, "q": NODE2VEC_Q},
            "deepwalk": {"p": 1.0, "q": 1.0},
            "statistics": {
                "friedman": True,
                "all_pairwise_wilcoxon": True,
                "holm_bonferroni": True,
                "rank_biserial_effect_size": True,
                "bootstrap_ci": "5,000-resample bootstrap over five repeat means; descriptive because the number of repeat blocks is limited",
                "alpha": ALPHA,
                "scope": "exploratory; inferential blocks are repeat means, not individual folds",
            },
        },
        "human_validation": {
            "performed": False,
            "statement": "No human ratings were supplied or fabricated. Deterministic unscored topic and NER audit templates are generated for optional later annotation.",
        },
        "environment": {
            "python": sys.version,
            "platform": platform.platform(),
            "processor": platform.processor(),
            "device": DEVICE,
            "torch_version": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_version": torch.version.cuda,
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "package_versions": PACKAGE_VERSIONS,
            "expected_package_versions": EXPECTED_PACKAGE_VERSIONS,
            "package_version_mismatches": PACKAGE_VERSION_MISMATCHES,
            "strict_package_versions": STRICT_PACKAGE_VERSIONS,
            "huggingface_model_revisions": MODEL_REVISIONS,
            "stanza_resource_file_sha256": stanza_resource_hashes,
            "executed_notebook_or_script_source_sha256": executed_source_sha256(),
        },
        "limitations": limitations.to_dict(orient="records"),
    }


def build_final_evidence() -> dict[str, Any]:
    return {
        "protocol": build_protocol(),
        "integrity_checks": integrity_checks,
        "null_and_provenance_audit": null_audit,
        "preprocessing_outcomes": pd.DataFrame(sorted(preprocessing_reason_counts.items()), columns=["status", "documents"]),
        "split_distribution": split_table,
        "sample_provenance_50000": sample_provenance,
        "token_length_summary": valid_sample["token_count"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).reset_index().rename(columns={"index": "statistic", "token_count": "value"}),
        "topic_data_summary": topic_data_summary,
        "topic_model_runs_all": all_runs,
        "topic_model_core_runs": core_runs,
        "topic_model_aggregate": aggregate,
        "topic_model_core_aggregate": core_aggregate,
        "topic_model_primary_common_k": common_k_results,
        "topic_model_metric_optima": metric_optima,
        "topic_model_core_pareto_front": core_pareto_front,
        "topic_best_configurations": best_by_model_display,
        "representative_topics": representative_topics,
        "all_topic_words": all_topic_words,
        "topic_human_audit_template_unscored": topic_audit_template,
        "topic_friedman_tests": friedman_results,
        "topic_pairwise_tests": pairwise_topic_results,
        "bertopic_encoder_runs": bertopic_encoder_runs,
        "bertopic_encoder_aggregate": bertopic_encoder_aggregate,
        "bertopic_encoder_tests": bertopic_encoder_tests,
        "preprocessing_ablation_runs": ablation_runs,
        "preprocessing_ablation_summary": ablation_summary,
        "preprocessing_ablation_friedman_tests": ablation_friedman_results,
        "preprocessing_ablation_pairwise_tests": ablation_pairwise_tests,
        "ner_summary": ner_summary,
        "ner_type_distribution": ner_type_summary,
        "ner_human_audit_template_unscored": ner_audit_template,
        "ner_top_surfaces_overall": top_entity_surfaces,
        "ner_top_surfaces_by_type": top_entity_surfaces_by_type,
        "ner_test_top_surfaces_overall": top_entity_surfaces_test,
        "ner_test_top_surfaces_by_type": top_entity_surfaces_by_type_test,
        "ner_surface_frequency_all": entity_surface_frequency_all,
        "ner_document_summary": ner_document_summary,
        "ner_errors": ner_errors,
        "entity_inventory_5000": entity_inventory,
        "graph_summary": graph_summary,
        "graph_type_distribution": graph_type_summary,
        "graph_degree_distribution": graph_degree_distribution,
        "graph_edge_weight_distribution": graph_edge_weight_distribution,
        "graph_component_distribution": graph_component_distribution,
        "graph_topic_vector_sensitivity": graph_topic_vector_summary,
        "link_prediction_folds": link_folds,
        "link_prediction_fold_assignments": link_fold_assignments,
        "link_prediction_evaluation_diagnostics": link_evaluation_diagnostics,
        "link_prediction_repeat_means": link_repeat_means,
        "link_prediction_summary": link_summary,
        "link_prediction_friedman_tests": link_friedman_results,
        "link_prediction_pairwise_tests": link_pairwise_tests,
        "link_prediction_main_comparisons": main_vs_baselines,
        "claim_ready_summary": claim_ready_summary,
        "explicit_limitations": limitations,
    }


def dataframe_markdown(frame: pd.DataFrame, columns: Sequence[str] | None = None, max_rows: int = 50) -> str:
    table = frame.copy()
    if columns is not None:
        table = table[list(columns)]
    if len(table) > max_rows:
        table = table.head(max_rows)
    # Try to use tabulate, fallback to string representation if not available
    try:
        return table.to_markdown(index=False, floatfmt=".4f")
    except ImportError:
        # Fallback to simple string representation if tabulate is not installed
        return table.to_string(index=False, float_format="%.4f")


def build_markdown_report() -> str:
    topic_columns = [
        "model", "encoder", "k", "c_v_mean", "c_v_sd", "c_npmi_mean", "c_npmi_sd",
        "diversity_mean", "distinctness_mean", "stability", "valid_topic_fraction_mean",
    ]
    link_columns = [
        "method", "fold_evaluations", "independent_repeat_blocks",
        "roc_auc_mean", "roc_auc_repeat_sd",
        "average_precision_mean", "average_precision_repeat_sd",
    ]
    lines = [
        "# Persian Topic Modeling and Entity-Association Graph: Reproducibility Report",
        "",
        f"- Run finished (UTC): {datetime.now(timezone.utc).isoformat()}",
        f"- Source dataset SHA-256: `{dataset_sha256}`",
        f"- Raw source rows scanned: {raw_row_count:,}",
        f"- Unique Persian-rich rows after hash deduplication: {len(unique_candidate_meta):,}",
        f"- Working sample: {len(sample):,}; train: {len(train):,}; test: {len(test):,}",
        f"- Descriptive topic grid: {list(TOPIC_COUNTS)}; statistical grid: {list(STATISTICAL_TOPIC_COUNTS)}; seeds: {list(SEEDS)}",
        f"- Core topic-model runs: {expected_core_runs}; ParsBERT sensitivity runs: {expected_parsbert_extra_runs}",
        "",
        "## Primary common-K topic-model comparison",
        "",
        dataframe_markdown(common_k_results, topic_columns),
        "",
        "## Coherence-maximizing configurations (metric-specific; not overall winners)",
        "",
        dataframe_markdown(best_by_model_display, topic_columns),
        "",
        "## Pareto-efficient core configurations",
        "",
        dataframe_markdown(core_pareto_front, topic_columns, max_rows=100),
        "",
        "## Named-entity recognition summary",
        "",
        dataframe_markdown(ner_summary),
        "",
        "## Entity graph summary",
        "",
        dataframe_markdown(graph_summary),
        "",
        "## Leakage-controlled link-prediction summary",
        "",
        dataframe_markdown(link_summary, link_columns),
        "",
        "## Claim-ready statements",
        "",
        dataframe_markdown(claim_ready_summary, max_rows=100),
        "",
        "## Explicit limitations",
        "",
        dataframe_markdown(limitations, max_rows=100),
        "",
        "## Integrity checks",
        "",
        dataframe_markdown(pd.DataFrame([{"check": key, "passed": value} for key, value in integrity_checks.items()]), max_rows=200),
        "",
        "Full run-level values, topic words, provenance, fold assignments, pair hashes, diagnostics, and tests are stored in `q1_final_evidence.json`.",
    ]
    return "\n".join(lines)


def write_evidence_files() -> None:
    evidence = build_final_evidence()
    FINAL_JSON.write_text(
        json.dumps(sanitize_json(evidence), ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    FINAL_REPORT.write_text(build_markdown_report(), encoding="utf-8")


# Draft write and physical validation, then a final write containing completed checks.
write_evidence_files()
with FINAL_JSON.open("r", encoding="utf-8") as handle:
    validated_json = json.load(handle)
if not isinstance(validated_json, dict):
    raise RuntimeError("Final JSON is not a top-level object.")
if not FINAL_REPORT.exists() or FINAL_REPORT.stat().st_size == 0:
    raise RuntimeError("Markdown report was not created.")

# Temporary resources are not public evidence files.
shutil.rmtree(TEMP_DIR, ignore_errors=True)
public_files = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())

integrity_checks.update({
    "json_file_created_and_valid": FINAL_JSON.exists() and FINAL_JSON.stat().st_size > 0,
    "markdown_report_created_and_valid": FINAL_REPORT.exists() and FINAL_REPORT.stat().st_size > 0,
    "output_directory_contains_only_two_public_files": public_files == sorted([FINAL_JSON.name, FINAL_REPORT.name]),
})
failed_final = [name for name, passed in integrity_checks.items() if not passed]
if failed_final:
    raise RuntimeError(f"Final integrity checks failed: {failed_final}")

# Rewrite files so their integrity sections contain the final passed state.
write_evidence_files()
with FINAL_JSON.open("r", encoding="utf-8") as handle:
    json.load(handle)

print("\nIntegrity checks — all must be True")
display(pd.DataFrame([{"check": key, "passed": value} for key, value in integrity_checks.items()]))
print(f"\nPrimary publication table: common K={PRIMARY_COMMON_K}")
display(common_k_results)
print("\nMetric-specific coherence maxima — interpret with the Pareto table")
display(best_by_model_display)
print("\nPublication-ready link-prediction table")
display(link_summary)
print("\nClaim-ready statements")
display(claim_ready_summary)
print("\nExplicit limitations and inference scope")
display(limitations)

heading("9. Final files")
json_sha256 = sha256_file(FINAL_JSON)
report_sha256 = sha256_file(FINAL_REPORT)
elapsed_hours = (time.time() - START_TIME) / 3600
print("====================")
print("FINAL FILES")
print("====================")
print(f"JSON evidence: {FINAL_JSON}")
print(f"Markdown report: {FINAL_REPORT}")
print(f"JSON SHA-256: {json_sha256}")
print(f"Markdown SHA-256: {report_sha256}")
print(f"Dataset SHA-256: {dataset_sha256}")
print(f"Runtime: {elapsed_hours:.2f} hours")
print("\nAll integrity checks passed. No figures, dummy entities, fabricated topics, or fabricated statistical results were generated.")
print("Inference remains exploratory where stated and is not population-level generalization.")

[2026-07-24 07:01:52] Dataset: C:\Users\Amir\Desktop\My Desk\Desktop 20260506\Articles\Second article\FInilized\version 8 (last run)\data\wikipedia.csv
[2026-07-24 07:01:52] Output directory: C:\Users\Amir\Desktop\My Desk\Desktop 20260506\Articles\Second article\FInilized\version 8 (last run)\q1_final_run
[2026-07-24 07:01:52] Device: cpu



## 1. Full-dataset audit and sampling

[2026-07-24 07:01:57] Text column: content; title column: title


Pass 1/2: full corpus audit: 0it [00:00, ?it/s]

Pass 2/2: retrieve sampled documents: 0it [00:00, ?it/s]

...

,area,statement_type,claim_ready_statement,statistically_validated,scope
0,dataset,observation,"The complete source file contained 2,525,369 rows. After language eligibility screening and text-hash deduplication,...",None,Descriptive source-corpus audit.
1,sampling,methodological fact,"A proportional raw-length-stratified random sample of 50,000 documents was drawn from the full eligible corpus; the ...",None,Reproducible sampling procedure conditional on seed 42.
2,topic modeling,observation,"At the predeclared common topic count K=30, the four model families were compared under the same held-out corpus and...",False,Primary matched-K descriptive comparison; model-specific optima are secondary sensitivity analyses.
3,topic modeling,observation,"At K=30, BERTopic had the highest mean C_v (0.5460); NPMI, diversity, distinctness, and stability are reported besid...",False,Primary common-K descriptive result with explicit multi-metric qualification.
4,topic modeling,observation,"When each core model was allowed to choose its own coherence-maximizing K, BERTopic achieved the highest mean C_v (0...",False,Descriptive metric-specific selection; no composite quality score is used.
5,topic modeling,exploratory inference,"Across the 27 matched K × seed configurations in the fixed statistical grid, NMF had the highest mean C_v (0.5312); ...",True,Exploratory paired inference across the fixed statistical K grid; model-specific maxima are descriptive only.
6,BERTopic encoder sensitivity,exploratory inference,"Under fixed BERTopic settings, mean-pooled ParsBERT minus multilingual MiniLM produced a mean C_v difference of 0.00...",True,"Conditional on mean pooling, truncation length, UMAP, KMeans, and c-TF-IDF parameters."
7,NER,observation,"Stanza extracted 47,493 accepted training mentions and 45,947 accepted link-test mentions from 10,000 training and 1...",None,Typed normalized surface forms; no entity linking or canonical disambiguation.
8,entity graph,observation,"The training-only graph retained 5,000 typed surface-form nodes and 57,696 document co-occurrence edges.",None,"Entity association graph, not a multi-relational knowledge graph."
9,link prediction,observation,The predeclared Topic-K200 method achieved mean ROC-AUC 0.6387 and mean average precision 0.6166; means and uncertai...,False,Descriptive repeated cross-validation result.



Explicit limitations and inference scope


,limitation,statement
0,Hash-based deduplication,Corpus-scale deduplication uses pandas' stable 64-bit text hash for tractability. Collisions are extremely unlikely ...
1,Shared-source evaluation,"Preprocessing, topics, named entities, graph edges, and held-out links originate from the same Wikipedia source, alt..."
2,Domain specificity,"Results describe Persian Wikipedia and do not establish population-level generalization to social media, news, conve..."
3,CTM training budget,"CTM uses the official CombinedTM.fit loop for ten epochs and the shared 10,000-term vocabulary; conclusions remain c..."
4,ParsBERT pooling,ParsBERT is a masked-language-model checkpoint converted to document embeddings through mean pooling; it is not a Pe...
5,Encoder/parameter conditionality,"BERTopic conclusions are conditional on the selected encoders, 256-token truncation, UMAP, KMeans, Persian whitespac..."
6,No human validation,This pipeline contains no human topic-interpretability or edge-validity annotations and makes no inter-rater-reliabi...
7,Three-seed uncertainty,Topic-model means and seed-bootstrap intervals are based on only three predeclared random seeds; they characterize i...
8,Repeated-fold inference,Repeated cross-validation folds overlap; Wilcoxon tests and bootstrap intervals are exploratory rather than independ...
9,NER truncation,"NER processes at most the first 5,000 characters of each document; entities appearing later in long documents are no..."



## 9. Final files

FINAL FILES
JSON evidence: C:\Users\Amir\Desktop\My Desk\Desktop 20260506\Articles\Second article\FInilized\version 8 (last run)\q1_final_run\q1_final_evidence.json
Markdown report: C:\Users\Amir\Desktop\My Desk\Desktop 20260506\Articles\Second article\FInilized\version 8 (last run)\q1_final_run\q1_final_report.md
JSON SHA-256: ccb6715e88fedf75cd939a84a6fcda4c937123b80bfc67eb4fe034da57de7810
Markdown SHA-256: 29eca145d8ef9318e8703b431d9719e83e4d87f83402f223bf31b2afbc36f30e
Dataset SHA-256: 63fa7270b573b0b0dd72a46a8441fc8cc09cd958e524f88792f607ea65109406
Runtime: 10.32 hours

All integrity checks passed. No figures, dummy entities, fabricated topics, or fabricated statistical results were generated.
Inference remains exploratory where stated and is not population-level generalization.
